# SWOT Multi-Product Water Surface Elevation Pipeline — Pantanal Rivers

**Code used in:**

> Moreno-Oliveira, J., Fassoni-Andrade, A., Trigg, M. A., Moreira, D. M., & Novo,
> E. M. L. de M. *Multi-product validation of SWOT water surface elevation in
> the Pantanal Rivers*. Manuscript submitted to *Science of Remote Sensing*; in peer review.
>
> a. National Institute for Space Research (INPE), São José dos Campos, Brazil
> b. Institute of Geosciences, University of Brasília (UnB), Brasília, Brazil
> c. School of Civil Engineering, University of Leeds, Leeds, United Kingdom
> d. Geological Survey of Brazil, Rio de Janeiro, Brazil

Author of this code: **Jahdy Moreno-Oliveira**.

## What this notebook does

For a set of in-situ gauging stations (as points, with an associated water-body
polygon per station), this notebook:

1. **Downloads** all four SWOT Level-2 hydrology products for each station's
   area of interest and date range, via NASA `earthaccess`:
   - **PIXC** — `SWOT_L2_HR_PIXC` (pixel cloud)
   - **Raster** — `SWOT_L2_HR_Raster` (gridded WSE raster, 100 m)
   - **Node** and **Reach** — `SWOT_L2_HR_RiverSP` (river vector product)
2. **Processes** each product with its own product-specific logic (spatial
   filtering to the station's water body, SWOT quality-flag filtering) and
   reports a physically self-consistent WSE per overpass: for PIXC/Raster,
   rather than averaging or independently aggregating each attribute across
   many pixels/cells (which could mix values from different physical
   locations into one synthetic row), the pipeline locates the ONE pixel/cell
   whose WSE is the median value for that overpass — an exact rank, never an
   interpolation, always a real observation — and reports every other
   attribute (height, tides, sig0, quality flags, ...) from that SAME
   pixel/cell, the same way Node/Reach already report one feature's own
   attributes rather than an aggregate across many.
3. **Corrects Reach WSE for spatial offset**: the Reach product reports WSE at
   the *center* of a ~10 km river reach, not at the gauge location. We
   translate it to the gauge position along the river centerline using the
   reach slope (see `wse_corrected` section below for the exact formula and
   its reference).
4. **Cross-checks completeness**: compares, date by date, which overpasses
   produced valid data in each of the four products. Any date with valid data
   in at least one product but missing from another is re-downloaded and
   re-processed to confirm whether the gap is a genuine SWOT/processing
   outcome (e.g., filtered by quality, or the pass fell outside the water
   mask) or a recoverable gap — nothing is left unverified.
5. **Audits and de-duplicates** the final CSVs, so the released per-station
   tables never contain a duplicated overpass.

## Requirements

- Python ≥ 3.10, with `geopandas`, `pandas`, `numpy`, `xarray`, `rioxarray`,
  `earthaccess`, `shapely`, `pyproj`, `matplotlib`, and (for Raster) a working
  GDAL install with the `gdalmdimtranslate` command-line tool on `PATH`.
- A NASA Earthdata account configured in `~/.netrc` for `earthaccess`.
- Two input vector files (see `CONFIG` cell below):
  - a **station points** layer, one point per gauge, with a station-code field;
  - a **water-mask polygons** layer, one polygon per station delineating the
    water surface to sample (river channel / lake extent near the gauge).

> **Before running**: point `STATIONS_VECTOR` / `WATER_MASK_VECTOR` (in the
> `CONFIG` cell) at your own station-points and water-mask-polygons layers —
> by default these are `input/stations.gpkg` and `input/water_masks.gpkg`
> relative to the repository root. `OUTPUT_DIR` defaults to `output/` at
> the repository root. `STATION_CODES = None` processes every
> station found in `STATIONS_VECTOR`; set it to a list of codes to restrict
> the run to a subset (handy for a first, quick check before a full run).

## How to run

Every step is gated by a boolean flag in the last cell (`RUN_*`). Nothing in
this notebook blocks on interactive input — it is safe to "Run All" end to
end. Set `STATION_CODES = None` to process every station found in the stations
layer, or a list of codes to run a subset.

The pipeline is **resumable by construction**: every stage checks what
already exists on disk under `OUTPUT_DIR` before doing any work, and logs how
much it found (e.g. `"12 row(s) already saved; 3 new candidate(s)"`). Running
this notebook again against an environment that already has output from a
previous run — partial or complete — never re-downloads or re-processes a
granule/date that was already handled; it only fills in what's missing.

## Processing order

Blocks run in this order, and each one is a separate cell:

1. **PIXC** for every station, 2. **Raster** for every station, 3. **Node &
   Reach** for every station — one product at a time, across the whole
   station list, rather than looping station by station through all four
   products. Each product is self-contained (its own search query, download
   batching, and CSV), so finishing it everywhere before switching keeps
   progress easy to follow in the logs and avoids re-initializing
   per-product state (e.g. Raster's discovered tile patterns) on every
   switch.
4. **Cross-product completeness check** (Block 6) — only meaningful once all
   four products have been extracted for every station, since it compares
   them against each other.
5. **Duplicate audit** (Block 7) — the data should be de-duplicated *before*
   anything is derived from it.
6. **Reach WSE correction** (Block 5) — needs the final, de-duplicated
   Node/Reach tables, so it runs last.

A stage summary (rows per station per product) is printed after PIXC, after
Raster, after Node/Reach, and once more at the very end — so you can see how
the dataset evolves at each step instead of only at the finish line, and spot
a station stuck at 0 rows immediately rather than after the full run.

## Folder layout

Everything is written under `OUTPUT_DIR` (`output/` by default):

```
output/
├── summary_report.csv                  <- rows per station per product (Block 8)
└── Station_<code>/
    ├── pixc.csv                        <- one row per valid PIXC overpass
    ├── pixc_failures.csv               <- one row per rejected/failed PIXC granule, with why
    ├── pixc_maps/                      <- QC map PNG per valid overpass
    ├── pixc_temp/                      <- scratch space, safe to delete between runs
    │
    ├── raster.csv                      <- one row per valid Raster overpass
    ├── raster_maps/
    ├── raster_temp/
    ├── raster_valid_tiles.json         <- cached result of the one-off tile-overlap scan
    ├── raster_attempted.txt            <- granules already attempted (checked before re-downloading)
    │
    ├── reach.csv                       <- one row per valid Reach overpass (+ wse_corrected, Block 5)
    ├── node.csv                        <- one row per valid Node overpass
    ├── reach_vectors/, node_vectors/   <- matched-feature shapefile + QC map per overpass
    ├── reach_temp/, node_temp/
    ├── .processing_checkpoint.json     <- granules already handled for Node+Reach (any outcome)
    │
    ├── {product}_dates_checked.json    <- dates already verified by the completeness check (Block 6)
    └── *.bak / *.bak_<timestamp>       <- pre-overwrite backups (duplicate audit, WSE correction)
```

`*_temp/` folders only ever hold in-flight downloads/extractions; they are
cleaned automatically during a run and can always be safely deleted between
runs. Every other file is an input to resuming the pipeline — do not delete
them unless you intend to reprocess a station from scratch.


In [ ]:
# =============================================================================
# BLOCK 0 — IMPORTS, ENVIRONMENT, AND CONFIGURATION
# =============================================================================
import os
import re
import gc
import json
import time
import shutil
import zipfile
import warnings
import threading
import subprocess
from multiprocessing import cpu_count
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import geopandas as gpd
import shapely
from shapely.ops import transform as shp_transform
import xarray as xr
import rioxarray  # noqa: F401  (registers the .rio accessor used below)
from pyproj import Transformer
import earthaccess

import matplotlib
matplotlib.use('Agg')  # headless rendering — required when generating maps from worker threads
import matplotlib.pyplot as plt

try:
    import contextily as ctx
    HAS_CONTEXTILY = True
except ImportError:
    HAS_CONTEXTILY = False

try:
    import pyogrio  # noqa: F401
    VECTOR_ENGINE = 'pyogrio'
except ImportError:
    VECTOR_ENGINE = 'fiona'

warnings.filterwarnings('ignore')
os.environ.setdefault('CPL_LOG', '/dev/null')
os.environ.setdefault('GDAL_PAM_ENABLED', 'NO')
os.environ.setdefault('HDF5_USE_FILE_LOCKING', 'FALSE')
os.environ.setdefault('OMP_NUM_THREADS', '1')

# =============================================================================
# CONFIG — every path/parameter a new user needs to touch lives here.
# =============================================================================

# ---- Inputs ----------------------------------------------------------------
# Station points layer: one point per gauging station. Must contain the field
# named by STATION_CODE_FIELD (the station's unique code, e.g. an ANA — Agência
# Nacional de Águas — telemetric gauge code in the original study).
#
# Resolve paths from the repository root, whether Jupyter starts there or
# inside notebooks/river_validation. Do not commit local input datasets.
REPO_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'CITATION.cff').exists()), None)
if REPO_ROOT is None:
    raise RuntimeError('Open this notebook from inside a checkout of the repository.')
STATIONS_VECTOR = REPO_ROOT / 'input' / 'stations.gpkg'

# Water-mask polygons layer: one polygon per station, delineating the water
# surface to sample (the river channel or lake extent around the gauge). This
# is what actually constrains which SWOT pixels/nodes/reaches "belong" to a
# given station — the point above is only used as the reference location for
# distance/offset calculations (e.g. the Reach WSE correction below).
WATER_MASK_VECTOR = REPO_ROOT / 'input' / 'water_masks.gpkg'

STATION_CODE_FIELD = 'Codigo'

# If a station has no matching polygon in WATER_MASK_VECTOR, fall back to a
# circular buffer (in km) around its point instead of skipping it.
DEFAULT_SEARCH_BUFFER_KM = 2.0

# Keep generated products separate from version-controlled sources. Do not
# point this at an existing production directory without checking schemas.
OUTPUT_DIR = REPO_ROOT / 'output'

# ---- Study period ------------------------------------------------------------
DATE_START = '2023-03-31'
DATE_END = '2025-08-01'

# ---- SWOT product identifiers (Version D / Nominal science phase only) -----
SHORT_NAME_PIXC = 'SWOT_L2_HR_PIXC_D'
SHORT_NAME_RASTER = 'SWOT_L2_HR_Raster_D'
SHORT_NAME_RIVERSP = 'SWOT_L2_HR_RiverSP_D'

# ---- Quality-control thresholds ---------------------------------------------
# SWOT quality flags (node_q, reach_q, wse_qual, water_area_qual, wse_area_qual)
# are ordinal: 0=good, 1=suspect, 2=degraded, 3=bad. QUALITY_FLAG_MAX=2 keeps
# 0 and 1, discards 2 and 3.
QUALITY_FLAG_MAX = 2
WATER_FRAC_MIN = 0.7  # PIXC/Raster: keep pixels whose water fraction exceeds this
PIXC_CLASSIFICATION_KEEP = [3, 4, 5, 6, 7]  # PIXC 'classification': water-related classes only
RASTER_MIN_VALID_PIXELS = 2  # drop a Raster overpass whose water-body clip has <= this many valid pixels
MAX_MATCH_DISTANCE_KM = 1.0  # Node/Reach: max allowed distance between a matched river feature and the station
MIN_FAILED_ATTEMPTS_BEFORE_SKIP = 3  # PIXC: stop retrying a spatial tile after this many failures

# ---- Per-product metadata ----------------------------------------------------
# Single source of truth for each product's output file, how to read an
# overpass date out of it, and its natural key (the column(s) that uniquely
# identify one overpass/feature — used everywhere a duplicate must be
# rejected: at write time in Blocks 2-4 and 6, and in the Block 7 audit).
# 'key' is a granule/feature identity, deliberately NOT the calendar date —
# a single day can have more than one legitimate overpass (different SWOT
# pass), so collapsing by date alone would silently discard real data.
PRODUCT_SPEC = {
    'pixc':   {'csv': 'pixc.csv',   'date_column': 'date',           'date_format': 'iso',     'key': ['file']},
    'raster': {'csv': 'raster.csv', 'date_column': 'date',           'date_format': 'iso',     'key': ['file']},
    'node':   {'csv': 'node.csv',   'date_column': 'file_timestamp', 'date_format': 'compact', 'key': ['source_file', 'node_id']},
    'reach':  {'csv': 'reach.csv',  'date_column': 'file_timestamp', 'date_format': 'compact', 'key': ['source_file', 'reach_id']},
}

# ---- Parallelism --------------------------------------------------------------
# Station-level parallelism uses separate processes (CPU-bound geometry work:
# GDAL clips, overlays, shapefile I/O). Within a station, file-level
# parallelism uses threads (I/O-bound: NASA downloads release the GIL), so it
# can safely exceed the core count — tune IO_MULTIPLIER down if you see more
# download failures/timeouts in the logs (a sign of oversubscribing the
# Earthdata API), or up if workers sit idle waiting on downloads.
N_CORES = max(1, cpu_count() or 1)
IO_MULTIPLIER = 3
N_STATION_PROCESSES = max(1, min(N_CORES, round(N_CORES ** 0.5)))
N_FILE_THREADS = max(1, (N_CORES * IO_MULTIPLIER) // N_STATION_PROCESSES)
RASTER_DOWNLOAD_CHUNK = 60  # files fetched per download batch (kept conservative to avoid request timeouts)


def station_folder(station_code) -> Path:
    code = str(station_code).replace('.0', '')
    folder = OUTPUT_DIR / f'Station_{code}'
    folder.mkdir(parents=True, exist_ok=True)
    return folder


def log(msg):
    print(msg, flush=True)


def login_earthaccess():
    """Authenticates with NASA Earthdata strictly from `~/.netrc` — never
    falls back to earthaccess's interactive (ipywidgets-based) login prompt,
    which fails with a cryptic 'Error displaying widget: model not found' in
    notebook front ends that don't have the widget comm channel set up (and
    would block a non-interactive run anyway). Configure `~/.netrc` once
    (see earthaccess's docs) and every call in this notebook reuses it."""
    earthaccess.login(strategy='netrc')


In [ ]:
# =============================================================================
# BLOCK 1 — SHARED INFRASTRUCTURE (checkpointing, downloads, incremental CSV
# writes, cleanup). Used by all four product pipelines below.
# =============================================================================


class ProcessingCheckpoint:
    """Tracks which SWOT granules have already been *technically* handled for a
    given station/product, independent of whether they produced usable data.

    A granule that opens correctly and gets a filter decision (e.g. "outside
    the water mask", "quality-filtered") is recorded here even when it
    contributes zero rows to the output CSV — so it is never re-downloaded on
    a later run. Granules that fail for a transient reason (network error,
    corrupted download) are deliberately NOT recorded, so they are retried
    automatically next time.
    """

    def __init__(self, pasta_estacao):
        self.pasta_base = Path(pasta_estacao)
        self.state_file = self.pasta_base / '.processing_checkpoint.json'
        self.done = self._load()

    def _load(self):
        if self.state_file.exists():
            try:
                return set(json.loads(self.state_file.read_text()))
            except Exception:
                return set()
        return set()

    def is_done(self, granule_id):
        return str(granule_id).replace('.zip', '') in self.done

    def mark_done(self, granule_ids):
        if not granule_ids:
            return
        self.done.update(str(g).replace('.zip', '') for g in granule_ids)
        tmp = self.state_file.with_suffix('.tmp')
        tmp.write_text(json.dumps(sorted(self.done)))
        tmp.replace(self.state_file)

    def unmark(self, granule_ids):
        """Forces the given granules to be reprocessed on the next run — used
        when a downstream integrity check finds their data is missing from the
        CSV despite being marked done (see `sanitize_existing_csv`)."""
        ids = {str(g).replace('.zip', '') for g in granule_ids}
        removed = self.done & ids
        if not removed:
            return removed
        self.done -= ids
        tmp = self.state_file.with_suffix('.tmp')
        tmp.write_text(json.dumps(sorted(self.done)))
        tmp.replace(self.state_file)
        return removed


def extract_datetime_from_filename(name):
    """SWOT granule filenames all embed a `..._YYYYMMDDTHHMMSS_...` start-time
    token. Used by PIXC and Raster for their 'date' column (Node/Reach keep
    the compact token as-is in 'file_timestamp' — see PRODUCT_SPEC)."""
    m = re.search(r'_(\d{8}T\d{6})_', str(name))
    if not m:
        return None
    d = m.group(1)
    return f'{d[:4]}-{d[4:6]}-{d[6:8]} {d[9:11]}:{d[11:13]}:{d[13:15]}'


def print_progress(prefix, done, total, extra=''):
    if done % 10 == 0 or done == total:
        suffix = f' | {extra}' if extra else ''
        log(f'   {prefix}: {done}/{total}{suffix}')


def read_csv_row_count(csv_path):
    p = Path(csv_path)
    if not p.exists():
        return 0
    try:
        return len(pd.read_csv(p, usecols=[0], engine='python', on_bad_lines='skip'))
    except Exception:
        return None


def plot_water_mask(ax, mask_gdf, crs, plot_crs='EPSG:3857', zorder=1, **style):
    """Reprojects the water mask to `plot_crs` and plots it. `style` defaults
    to a filled polygon; pass e.g. facecolor='none' for an outline (used when
    the mask is drawn on top of another raster/scatter layer). `plot_crs`
    must match whatever CRS the rest of the figure (basemap, other layers) is
    already in — usually Web Mercator, except Raster's map, which falls back
    to the native raster CRS if reprojecting the grid itself ever fails."""
    gs = gpd.GeoSeries(list(mask_gdf.geometry), crs=crs)
    (gs if str(crs) == str(plot_crs) else gs.to_crs(plot_crs)).plot(ax=ax, zorder=zorder,
        **({'color': 'cyan', 'alpha': 0.2, 'edgecolor': 'blue', 'linewidth': 1} | style))


def plot_station_marker(ax, station_gdf, crs, plot_crs='EPSG:3857', zorder=10, **style):
    gs = gpd.GeoSeries([station_gdf.geometry.iloc[0]], crs=crs)
    (gs if str(crs) == str(plot_crs) else gs.to_crs(plot_crs)).plot(ax=ax, zorder=zorder,
        **({'color': 'yellow', 'marker': '*', 'markersize': 200, 'edgecolor': 'black'} | style))


def add_satellite_basemap(ax, crs=None, **kwargs):
    if not HAS_CONTEXTILY:
        return
    try:
        if crs is not None:
            ctx.add_basemap(ax, crs=crs, source=ctx.providers.Esri.WorldImagery, attribution=False, **kwargs)
        else:
            ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery, attribution=False, **kwargs)
    except Exception:
        pass


def save_figure_safely(fig, out_path):
    try:
        fig.savefig(out_path, dpi=100, bbox_inches='tight', facecolor='white')
    except Exception:
        pass
    finally:
        plt.close(fig)
        plt.close('all')


def run_batched_download_pipeline(candidates, temp_dir, batch_size, n_workers, process_fn, on_result,
                                   file_suffix='.nc', label='', on_batch_start=None):
    """Shared download-then-process loop used by PIXC and Raster: splits
    `candidates` into batches, downloads each batch, runs `process_fn(path)`
    over the resulting files in a thread pool, calls `on_result(path, result)`
    for every completed file, then deletes the downloaded file regardless of
    outcome. Node/Reach does not use this — it downloads one file at a time
    from inside its worker (see `robust_download`), a different access
    pattern that batching would not help.

    `on_batch_start(batch)`, if given, runs before each batch is downloaded —
    e.g. to record every granule in the batch as "attempted" up front, so a
    download that hangs or fails silently can't cause the same batch to be
    retried forever.
    """
    total = len(candidates)
    done = 0
    for i in range(0, total, batch_size):
        batch = candidates[i:i + batch_size]
        if on_batch_start is not None:
            on_batch_start(batch)
        files = download_with_retries(batch, temp_dir)
        matched = [Path(f) for f in files if Path(f).suffix == file_suffix]
        for other in [Path(f) for f in files if Path(f).suffix != file_suffix]:
            try:
                other.unlink()
            except Exception:
                pass
        if not matched:
            done += len(batch)
            print_progress(label, done, total)
            continue

        def _process_and_clean(path):
            try:
                return process_fn(path)
            finally:
                try:
                    path.unlink()
                except Exception:
                    pass

        with ThreadPoolExecutor(max_workers=max(1, min(n_workers, len(matched)))) as executor:
            futures = {executor.submit(_process_and_clean, f): f for f in matched}
            for future in as_completed(futures):
                path = futures[future]
                try:
                    result = future.result()
                except Exception as e:
                    result = {'file': path.name, 'status': f'error_{type(e).__name__}: {str(e)[:150]}'}
                on_result(path, result)
                done += 1
                print_progress(label, done, total)
        gc.collect()


def robust_download(granule, dest_dir, filename, max_attempts=3):
    """Downloads a single granule via direct HTTPS (using the authenticated
    earthaccess session) with a couple of retries on transient network
    errors. Used by the Node/Reach pipeline, where files are downloaded one at
    a time inside worker threads (earthaccess.download() batches poorly for
    that access pattern)."""
    dest_path = Path(dest_dir) / filename
    try:
        if hasattr(granule, 'data_links'):
            urls = granule.data_links(access='external') or granule.data_links(access='on_prem')
            url = urls[0]
        elif isinstance(granule, str):
            url = granule
        else:
            return False
    except Exception:
        return False

    session = earthaccess.get_requests_https_session()
    wait_seconds = [5, 10]
    for attempt in range(max_attempts):
        try:
            with session.get(url, stream=True, timeout=30) as r:
                r.raise_for_status()
                with open(dest_path, 'wb') as f:
                    for chunk in r.iter_content(chunk_size=8192):
                        f.write(chunk)
            if dest_path.exists() and dest_path.stat().st_size > 1000:
                return True
        except Exception as e:
            if dest_path.exists():
                try:
                    dest_path.unlink()
                except Exception:
                    pass
            time.sleep(wait_seconds[min(attempt, len(wait_seconds) - 1)])
            if 'NameResolution' in str(e):
                try:
                    session = earthaccess.get_requests_https_session()
                except Exception:
                    pass
    return False


def download_with_retries(granules, dest_dir, max_attempts=3):
    """Batch download via earthaccess.download(), retried on transient errors.
    Used by PIXC and Raster, where earthaccess's own batching/parallelism is
    used directly."""
    for attempt in range(max_attempts):
        try:
            return earthaccess.download(granules, str(dest_dir))
        except Exception as e:
            wait = (attempt + 1) * 5
            log(f'   download error (attempt {attempt + 1}/{max_attempts}): {e} -- waiting {wait}s')
            time.sleep(wait)
    return []


def append_rows_aligned(csv_path, rows, lock=None, dedup_key=None):
    """Appends one or more rows to a per-station CSV, aligning them to the
    header already on disk (extra columns dropped, missing ones become NaN),
    and returns how many rows were actually written. `rows` may be a single
    dict, a list of dicts, or a DataFrame.

    Without this alignment, two batches of the same product with slightly
    different attribute sets (e.g. different SWORD schema versions across
    passes) can silently produce a ragged CSV, or — worse — permanently lock
    the file's header to whichever batch was written first. If that first
    batch happened to be missing the scientific columns (wse, node_q, ...),
    every later, valid batch would be silently truncated to match. Aligning
    every write to the existing header (once the header exists) prevents
    both failure modes; the WSE/quality columns are always produced by our
    own extraction code, never inferred from the raw shapefile, so they never
    depend on which batch happened to arrive first.

    If `dedup_key` is given (the column name(s) that uniquely identify one
    overpass/feature — see PRODUCT_SPEC), this is also the single choke point
    that keeps a granule already present in the CSV from ever being written
    to it again: any incoming row whose key already exists on disk is
    dropped before the write. This is what makes duplicate rows structurally
    impossible rather than something the Block 7 audit has to clean up after
    the fact — checkpoints (Block 1) already stop a granule from being
    *reprocessed*, this stops it from being *re-added* even if it somehow
    were (e.g. two different code paths, main extraction and the Block 6
    recovery, both concluding the same overpass is missing and both trying
    to add it).
    """
    df_new = rows if isinstance(rows, pd.DataFrame) else pd.DataFrame([rows] if isinstance(rows, dict) else rows)
    if df_new.empty:
        return 0
    csv_path = Path(csv_path)

    def _write():
        header = not csv_path.exists()
        payload = df_new
        if not header:
            try:
                existing_cols = list(pd.read_csv(csv_path, nrows=0).columns)
                payload = df_new.reindex(columns=existing_cols)
            except Exception:
                pass
            if dedup_key and all(c in payload.columns for c in dedup_key):
                try:
                    existing_keys = pd.read_csv(csv_path, usecols=dedup_key, engine='python', on_bad_lines='skip')
                    seen = set(map(tuple, existing_keys.astype(str).values))
                    is_new = ~payload[dedup_key].astype(str).apply(tuple, axis=1).isin(seen)
                    payload = payload[is_new]
                except Exception:
                    pass
        if payload.empty:
            return 0
        payload.to_csv(csv_path, mode='a', header=header, index=False)
        return len(payload)

    if lock is not None:
        with lock:
            return _write()
    return _write()


def deduplicate_csv(csv_path, subset=None):
    """One dedup pass over a station/product CSV — cheap cleanup for the
    duplicate rows that incremental appends can accumulate (e.g. the same
    overpass written twice by an interrupted-and-resumed run)."""
    path = Path(csv_path)
    if not path.exists():
        return
    try:
        df = pd.read_csv(path, engine='python', on_bad_lines='skip')
        before = len(df)
        df = df.drop_duplicates()
        if subset and all(c in df.columns for c in subset):
            df = df.drop_duplicates(subset=subset, keep='last')
        if len(df) != before:
            df.to_csv(path, index=False)
    except Exception:
        pass


def _safe_rglob(path, pattern):
    """Like Path.rglob, but tolerant of the directory tree changing under it
    mid-walk. rglob() recurses lazily, so if a concurrent thread deletes a
    directory (e.g. its own per-granule extraction folder, cleaned up in
    `process_river_granule`'s `finally`) after rglob has queued it for
    descent but before it actually scans it, the plain version raises
    FileNotFoundError and aborts the whole walk. Used here specifically
    because this function scans a temp_dir that OTHER threads are actively
    creating/deleting subdirectories in while this runs — see `only_old`."""
    try:
        it = path.rglob(pattern)
        while True:
            try:
                yield next(it)
            except StopIteration:
                return
            except (FileNotFoundError, NotADirectoryError):
                continue
    except (FileNotFoundError, NotADirectoryError):
        return


def cleanup_stale_extract_dirs(root_dir, only_old=False, max_age_seconds=600):
    """Removes leftover ZIP-extraction directories. When `only_old` is True
    (used for periodic cleanup *during* a run, with several threads
    downloading/extracting concurrently for the same station), only
    directories older than `max_age_seconds` are removed — otherwise a
    cleanup pass could delete a directory another thread is still reading
    from, mid-extraction. The unconditional variant (only_old=False) is only
    safe to call before any worker thread has started."""
    root = Path(root_dir)
    if not root.exists():
        return
    now = time.time()
    for d in _safe_rglob(root, 'ext_*'):
        try:
            if not d.is_dir():
                continue
            if only_old and (now - d.stat().st_mtime) < max_age_seconds:
                continue
        except (FileNotFoundError, OSError):
            continue
        shutil.rmtree(d, ignore_errors=True)

    for z in _safe_rglob(root, '*.zip'):
        if 'map_vector' in str(z):
            continue
        corrupt = False
        try:
            if z.stat().st_size == 0:
                corrupt = True
            else:
                with zipfile.ZipFile(z, 'r') as zf:
                    if zf.testzip() is not None:
                        corrupt = True
        except Exception:
            corrupt = True
        if corrupt:
            try:
                z.unlink()
            except Exception:
                pass


def load_verified_dates(pasta_estacao, product):
    """Loads the persistent per-station, per-product cache of dates that have
    already been checked by the completeness audit (Block 4), regardless of
    outcome. Re-running the audit never repeats work for a date that is
    already in here — see `run_completeness_check` for how entries are added.
    """
    p = Path(pasta_estacao) / f'{product}_dates_checked.json'
    if not p.exists():
        return {}
    try:
        return json.loads(p.read_text())
    except Exception:
        return {}


def save_verified_dates(pasta_estacao, product, data):
    p = Path(pasta_estacao) / f'{product}_dates_checked.json'
    try:
        p.write_text(json.dumps(data, ensure_ascii=False, indent=0))
    except Exception:
        pass


def is_transient_status(status):
    """A 'transient' status (network/download/processing error) must NOT be
    cached as a final answer — it has to be retried on the next run. Every
    other status (recoverable or not) is conclusive and gets cached."""
    return str(status).startswith('error_') or str(status).startswith('failed_')


In [ ]:
# =============================================================================
# BLOCK 2 — PIXC (pixel cloud) EXTRACTION
# =============================================================================
# For every PIXC granule intersecting a station's water mask, this keeps only
# water-classified, high-quality, high-water-fraction pixels, converts height
# to WSE (geoid + tide corrections applied), and reports the per-overpass
# median WSE together with a set of ancillary per-pixel means used for QC.

_water_mask_prep_cache = {}  # id(gdf) -> (weakref, (union_geom, minx, miny, maxx, maxy))


def _extract_pass_id(filename):
    m = re.search(r'PIXC_\d+_(\d+)_\d+[A-Z]?_', str(filename))
    return m.group(1) if m else None


def _extract_spatial_id(filename):
    m = re.search(r'PIXC_[\w.]+\d+_(\d+_\d+[A-Z]?)_', str(filename))
    return m.group(1) if m else None


def _normalize_categorical(value):
    try:
        if value is None:
            return None
        if isinstance(value, (bytes, np.bytes_)):
            value = value.decode('utf-8', errors='ignore')
        if isinstance(value, np.generic):
            value = value.item()
        if isinstance(value, str):
            value = value.strip().replace("'", '').replace('"', '')
            return None if value == '' or value.lower() in {'nan', 'none'} else value
        if pd.isna(value):
            return None
        if isinstance(value, (int, np.integer)):
            return int(value)
        if isinstance(value, (float, np.floating)):
            return None if not np.isfinite(value) else (int(value) if float(value).is_integer() else float(value))
        return str(value).strip()
    except Exception:
        return None


def _dominant_categorical_value(arr):
    try:
        if arr is None or len(arr) == 0:
            return None
        cleaned = [v for v in (_normalize_categorical(x) for x in np.asarray(arr, dtype=object).ravel()) if v is not None]
        if not cleaned:
            return None
        from collections import Counter
        return max(Counter(cleaned).items(), key=lambda kv: (kv[1], str(kv[0])))[0]
    except Exception:
        return None


def _prepared_water_mask_4326(water_mask_gdf):
    # The water mask is the same for every granule of a station, but a PIXC
    # swath covers a much larger area than the mask — reprojecting every pixel
    # to the mask's CRS before testing containment wastes most of that work on
    # points that get discarded immediately. Instead we reproject the (small,
    # fixed) mask to EPSG:4326 once per station and test raw lat/lon directly.
    import weakref
    key = id(water_mask_gdf)
    cached = _water_mask_prep_cache.get(key)
    if cached is not None:
        ref, result = cached
        if ref() is water_mask_gdf:
            return result
    gdf_4326 = water_mask_gdf if str(water_mask_gdf.crs).upper() in ('EPSG:4326', 'OGC:CRS84') else water_mask_gdf.to_crs('EPSG:4326')
    union = gdf_4326.geometry.union_all() if hasattr(gdf_4326.geometry, 'union_all') else gdf_4326.geometry.unary_union
    minx, miny, maxx, maxy = gdf_4326.total_bounds
    result = (union, float(minx), float(miny), float(maxx), float(maxy))
    _water_mask_prep_cache[key] = (weakref.ref(water_mask_gdf), result)
    return result


def _points_inside_mask(lon_arr, lat_arr, union_geom):
    if hasattr(shapely, 'contains_xy'):
        return shapely.contains_xy(union_geom, lon_arr, lat_arr)
    from shapely.vectorized import contains as sv_contains
    return sv_contains(union_geom, lon_arr, lat_arr)


def _render_pixc_map(lat, lon, wse, water_mask_gdf, station_gdf, out_path, station_code, wse_median, n_pixels):
    try:
        fig, ax = plt.subplots(figsize=(10, 8), dpi=100)
        points = gpd.GeoDataFrame({'wse': wse}, geometry=gpd.points_from_xy(lon, lat), crs='EPSG:4326').to_crs('EPSG:3857')
        plot_water_mask(ax, water_mask_gdf, water_mask_gdf.crs, zorder=1)
        sc = ax.scatter(points.geometry.x, points.geometry.y, c=points['wse'], cmap='viridis', s=3, alpha=0.8, zorder=3)
        plt.colorbar(sc, ax=ax, label='WSE (m)', shrink=0.7)
        plot_station_marker(ax, station_gdf, water_mask_gdf.crs, zorder=5, color='red', markersize=150, marker='*')
        add_satellite_basemap(ax, zoom='auto')
        ax.set_title(f'Station {station_code} | WSE: {wse_median:.2f} m | {n_pixels} px', fontsize=10)
        ax.set_axis_off()
        plt.tight_layout()
        save_figure_safely(fig, out_path)
    except Exception:
        pass
    finally:
        gc.collect()


def process_pixc_granule(nc_path, water_mask_gdf, station_gdf, maps_dir, station_code, generate_map=True):
    """Extracts a single median WSE value (and QC ancillaries) for one PIXC
    granule, restricted to the pixels that fall inside the station's water
    mask. Returns a dict with a 'status' key; 'status' == 'ok' means the row
    is ready to be saved."""
    name = nc_path.name

    def empty(status):
        return {'file': name, 'date': extract_datetime_from_filename(name), 'pass_id': _extract_pass_id(name), 'status': status}

    ds = None
    try:
        if not nc_path.exists() or nc_path.stat().st_size < 10000:
            return empty('corrupted_or_empty_file')
        try:
            ds = xr.open_dataset(nc_path, engine='h5netcdf', group='pixel_cloud')
        except Exception:
            try:
                ds = xr.open_dataset(nc_path, engine='h5netcdf')
            except Exception:
                return empty('netcdf_open_error')

        lat_var = 'latitude' if 'latitude' in ds.variables else 'lat'
        lon_var = 'longitude' if 'longitude' in ds.variables else 'lon'
        if lat_var not in ds.variables or 'height' not in ds.variables:
            ds.close()
            return empty('no_coordinates')

        lat = ds[lat_var].values.ravel()
        lon = ds[lon_var].values.ravel()
        n = min(len(lat), len(lon))
        if n == 0:
            ds.close()
            return empty('zero_pixels')
        lat, lon = lat[:n], lon[:n]

        mask = np.ones(n, dtype=bool)
        if 'classification' in ds.variables:
            mask &= np.isin(ds['classification'].values.ravel()[:n], PIXC_CLASSIFICATION_KEEP)
        if 'geolocation_qual' in ds.variables:
            # geolocation_qual is a BITWISE flag, not an ordinal scale: comparing
            # against a power of two ("< 4") means "no bit >= 2 is set", which is
            # the only bitwise-meaningful threshold here (an ordinal cut like
            # "< 3" would arbitrarily exclude the bit0+bit1 combination for no
            # physical reason).
            mask &= (ds['geolocation_qual'].values.ravel()[:n] < 4)

        if not np.any(mask):
            ds.close()
            return empty('quality_filtered')

        idx = np.where(mask)[0]
        lat_m, lon_m = lat[mask], lon[mask]

        union_geom, mx0, my0, mx1, my1 = _prepared_water_mask_4326(water_mask_gdf)
        bbox_ok = (lon_m >= mx0) & (lon_m <= mx1) & (lat_m >= my0) & (lat_m <= my1)
        if not np.any(bbox_ok):
            ds.close()
            return empty('outside_mask')
        idx_bbox = idx[bbox_ok]
        inside = _points_inside_mask(lon_m[bbox_ok], lat_m[bbox_ok], union_geom)
        if not np.any(inside):
            ds.close()
            return empty('outside_mask')
        idx_final = idx_bbox[inside]
        del lat, lon

        def get_num(varname, default=np.nan):
            if varname in ds.variables:
                v = ds[varname].values.ravel()
                if len(v) >= n:
                    return v[:n][idx_final].astype(np.float32)
                return np.full(len(idx_final), default, dtype=np.float32)
            return np.full(len(idx_final), default, dtype=np.float32)

        def get_cat(varname):
            if varname in ds.variables:
                v = ds[varname].values
                if v.size >= n:
                    return v.ravel()[:n][idx_final]
                if v.size == 1:
                    return np.full(len(idx_final), v.item())
            return np.array([None] * len(idx_final))

        height = get_num('height')
        geoid = get_num('geoid', 0)
        solid_earth_tide = get_num('solid_earth_tide', 0)
        load_tide_fes = get_num('load_tide_fes', 0)
        load_tide_got = get_num('load_tide_got', 0)
        pole_tide = get_num('pole_tide', 0)
        sig0 = get_num('sig0')
        water_frac = get_num('water_frac')
        layover_impact = get_num('layover_impact')
        lat_out = ds[lat_var].values.ravel()[:n][idx_final].astype(np.float32)
        lon_out = ds[lon_var].values.ravel()[:n][idx_final].astype(np.float32)

        wse = height - geoid - solid_earth_tide - load_tide_fes - load_tide_got - pole_tide

        wf_mask = ~np.isnan(water_frac) & (water_frac > WATER_FRAC_MIN)
        if not np.any(wf_mask):
            ds.close()
            return empty('water_fraction_filtered')
        arrays = [wse, height, geoid, solid_earth_tide, load_tide_fes, load_tide_got, pole_tide,
                  sig0, water_frac, layover_impact, lat_out, lon_out]
        wse, height, geoid, solid_earth_tide, load_tide_fes, load_tide_got, pole_tide, \
            sig0, water_frac, layover_impact, lat_out, lon_out = [a[wf_mask] for a in arrays]

        # No percentile-based outlier trimming here: the median is already
        # robust to outliers on its own, and instead of computing each
        # attribute's median INDEPENDENTLY (which would mix values from
        # different physical pixels into one synthetic, internally
        # inconsistent row), we locate the ONE pixel whose WSE is the median
        # value (exact rank, no interpolation -- always a real observed
        # pixel) and report ALL other attributes from that SAME pixel, the
        # same way Node/Reach already report one feature's own attributes.
        # "Lower median" index: for an even pixel count, this picks the
        # lower of the two central pixels instead of interpolating between
        # them (so the reported value always comes from a real pixel).
        order = np.argsort(wse)
        median_idx = order[(len(wse) - 1) // 2]
        wse_median = float(wse[median_idx])

        median_vars = ['phase_noise_std', 'dheight_dphase', 'dlatitude_dphase', 'dlongitude_dphase',
                       'dheight_droll', 'dheight_dbaseline', 'dheight_drange', 'darea_dheight',
                       'sig0_cor_atmos_model', 'height_cor_xover', 'model_dry_tropo_cor',
                       'model_wet_tropo_cor', 'iono_cor_gim_ka', 'coherent_power', 'power_plus_y',
                       'power_minus_y', 'x_factor_plus_y', 'x_factor_minus_y', 'water_frac_uncert',
                       'pixel_area', 'prior_water_prob']
        stats = {}
        for v in median_vars:
            arr = get_num(v)[wf_mask]
            val = arr[median_idx]
            stats[f'{v}_median'] = round(float(val), 6) if np.isfinite(val) else None

        cat_vars = ['polarization', 'transmit_side', 'classification', 'ancillary_surface_classification_flag',
                    'bright_land_flag', 'interferogram_qual', 'classification_qual', 'geolocation_qual',
                    'sig0_qual', 'pixc_line_qual']
        for v in cat_vars:
            arr = get_cat(v)[wf_mask]
            # No longer "the value most common across many pixels" -- it is
            # this ONE pixel's own value, so the column drops the old
            # '_dominant' suffix (which would now be misleading) and uses the
            # plain attribute name, matching Node/Reach.
            stats[v] = _normalize_categorical(arr[median_idx])

        ds.close()
        if generate_map:
            _render_pixc_map(lat_out, lon_out, wse, water_mask_gdf, station_gdf, maps_dir / f'map_{nc_path.stem}.png',
                              station_code, wse_median, len(wse))

        # wse_std stays the true standard deviation over ALL pixels passing
        # the filters (an overpass-level dispersion/noise indicator) -- the
        # one population-wide aggregate left on purpose; everything else
        # below comes from the SAME pixel (median_idx), like Node/Reach.
        result = {
            'file': name, 'date': extract_datetime_from_filename(name), 'pass_id': _extract_pass_id(name),
            'wse_median': round(wse_median, 4), 'wse_std': round(float(np.nanstd(wse)), 4),
            'lat_median': round(float(lat_out[median_idx]), 6), 'lon_median': round(float(lon_out[median_idx]), 6),
            'height_median': round(float(height[median_idx]), 4), 'geoid_median': round(float(geoid[median_idx]), 4),
            'solid_earth_tide_median': round(float(solid_earth_tide[median_idx]), 6),
            'load_tide_fes_median': round(float(load_tide_fes[median_idx]), 6),
            'load_tide_got_median': round(float(load_tide_got[median_idx]), 6),
            'pole_tide_median': round(float(pole_tide[median_idx]), 6),
            'sig0_median': round(float(sig0[median_idx]), 4),
            'water_frac_median': round(float(water_frac[median_idx]), 4),
            'layover_impact_median': round(float(layover_impact[median_idx]), 4),
            'n_pixels': len(wse), 'status': 'ok',
        }
        result.update(stats)
        return result

    except Exception as e:
        if ds is not None:
            try:
                ds.close()
            except Exception:
                pass
        return empty(f'error_{type(e).__name__}: {str(e)[:150]}')
    finally:
        plt.close('all')
        gc.collect()


def _pixc_search_candidates(bbox, already_ok_spatial_ids, failure_counts, already_seen_files, temporal):
    candidates, seen = [], set()
    try:
        results = earthaccess.search_data(short_name=SHORT_NAME_PIXC, bounding_box=bbox, temporal=temporal)
    except Exception:
        return candidates
    for item in results:
        granule_ur = item['umm']['GranuleUR']
        if granule_ur + '.nc' in already_seen_files or granule_ur in already_seen_files:
            continue
        spatial_id = _extract_spatial_id(granule_ur)
        if not spatial_id:
            continue
        if spatial_id in already_ok_spatial_ids or failure_counts.get(spatial_id, 0) < MIN_FAILED_ATTEMPTS_BEFORE_SKIP:
            m = re.search(r'_(\d{8}T\d{6})_', granule_ur)
            tag = m.group(1) if m else granule_ur
            if tag not in seen:
                candidates.append(item)
                seen.add(tag)
    return candidates


def _pixc_load_history(csv_valid, csv_failed):
    success_ids, failure_counts, seen_files = set(), {}, set()
    if csv_valid.exists():
        try:
            df = pd.read_csv(csv_valid, usecols=['file'], engine='python', on_bad_lines='skip')
            for f in df['file'].astype(str):
                seen_files.add(f)
                sid = _extract_spatial_id(f)
                if sid:
                    success_ids.add(sid)
        except Exception:
            pass
    if csv_failed.exists():
        try:
            df = pd.read_csv(csv_failed, usecols=['file'], engine='python', on_bad_lines='skip')
            for f in df['file'].astype(str):
                seen_files.add(f)
                sid = _extract_spatial_id(f)
                if sid:
                    failure_counts[sid] = failure_counts.get(sid, 0) + 1
        except Exception:
            pass
    return success_ids, failure_counts, seen_files


def process_station_pixc(station_code, bbox, water_mask_gdf, station_gdf):
    folder = station_folder(station_code)
    temp_dir = folder / 'pixc_temp'
    maps_dir = folder / 'pixc_maps'
    temp_dir.mkdir(exist_ok=True)
    maps_dir.mkdir(exist_ok=True)
    csv_valid = folder / 'pixc.csv'
    csv_failed = folder / 'pixc_failures.csv'
    existing_rows = read_csv_row_count(csv_valid) or 0

    success_ids, failure_counts, seen_files = _pixc_load_history(csv_valid, csv_failed)
    candidates = _pixc_search_candidates(bbox, success_ids, failure_counts, seen_files, (DATE_START, DATE_END))
    if not candidates:
        log(f'[{station_code}][PIXC] up to date, nothing new ({existing_rows} row(s) already saved).')
        return

    log(f'[{station_code}][PIXC] {existing_rows} row(s) already saved; {len(candidates)} new candidate granule(s).')
    saved, failed = 0, 0

    def _on_result(nc_path, res):
        nonlocal saved, failed
        if res.get('status') == 'ok':
            saved += append_rows_aligned(csv_valid, res, dedup_key=PRODUCT_SPEC['pixc']['key'])
        else:
            append_rows_aligned(csv_failed, {'file': res.get('file', nc_path.name), 'status': res.get('status')}, dedup_key=['file'])
            failed += 1

    run_batched_download_pipeline(
        candidates, temp_dir, batch_size=max(4, N_FILE_THREADS * 2), n_workers=N_FILE_THREADS,
        process_fn=lambda p: process_pixc_granule(p, water_mask_gdf, station_gdf, maps_dir, station_code),
        on_result=_on_result, label=f'[{station_code}][PIXC]',
    )

    deduplicate_csv(csv_valid, subset=['file'])
    log(f'[{station_code}][PIXC] done: +{saved} new row(s), {failed} filtered/failed. Total: {read_csv_row_count(csv_valid) or 0} row(s).')


In [ ]:
# =============================================================================
# BLOCK 3 — RASTER EXTRACTION
# =============================================================================
# The Raster product tiles the globe on a fixed UTM/MGRS-like grid, so most
# tiles a station's bounding box overlaps in the CMR search never actually
# intersect its (much smaller) water mask. We first discover which tile
# *patterns* are relevant for a station once (`discover_raster_tiles`), then
# only ever search/download those going forward.

RASTER_AUX_VARS = [
    'n_wse_pix', 'n_water_area_pix', 'n_sig0_pix', 'n_other_pix', 'dark_frac',
    'ice_clim_flag', 'ice_dyn_flag', 'layover_impact', 'wse_qual', 'wse_qual_bitwise',
    'wse_uncert', 'sig0', 'sig0_qual', 'sig0_qual_bitwise', 'sig0_uncert', 'water_area',
    'water_area_qual', 'water_area_qual_bitwise', 'water_area_uncert', 'water_frac',
    'water_frac_uncert', 'sig0_cor_atmos_model', 'height_cor_xover', 'geoid',
    'solid_earth_tide', 'load_tide_fes', 'load_tide_got', 'pole_tide',
    'model_dry_tropo_cor', 'model_wet_tropo_cor', 'iono_cor_gim_ka',
]


def _raster_extract_tile_pattern(name):
    m = re.search(r'_100m_(UTM\d+[A-Z]+)_', name)
    if m:
        return m.group(1)
    m = re.search(r'_(\d{2}[A-Z]{3})_', name)
    return m.group(1) if m else None


def _nc_to_tif(nc_path, tif_path, variable='wse'):
    """Uses the GDAL multidimensional translator to materialize one variable
    of a Raster granule as a georeferenced GeoTIFF — this gives us a real,
    correctly-referenced grid to clip against, which the raw NetCDF alone
    does not reliably provide."""
    try:
        if tif_path.exists():
            tif_path.unlink()
        cmd = ['gdalmdimtranslate', '-of', 'GTiff', '-co', 'COMPRESS=LZW', '-array', variable, str(nc_path), str(tif_path)]
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
        return result.returncode == 0
    except Exception:
        return False


def _raster_value_at_median_cell(ds, var, idx_flat):
    # Instead of computing each auxiliary variable's median INDEPENDENTLY
    # (which would mix values from different grid cells into one synthetic,
    # internally inconsistent row), this returns THIS SAME cell's own value
    # for `var` -- the cell already identified as the median-WSE cell for
    # this granule (idx_flat, computed once in process_raster_granule).
    if var not in ds:
        return None
    vals = ds[var].values.flatten()
    if idx_flat >= len(vals):
        return None
    val = vals[idx_flat]
    if not np.isfinite(val) or val <= -1.0e10:
        return None
    return round(float(val), 6)


def _render_raster_map(da_clip, crs_ref, geoms, station_geom, out_path, station_code, wse_median, n_pixels):
    try:
        valid = da_clip.where(da_clip > -1000)
        vals = valid.values.flatten()
        vals = vals[~np.isnan(vals)]
        if len(vals) == 0:
            return
        fig, ax = plt.subplots(figsize=(10, 10))
        if da_clip.rio.crs is None:
            da_clip.rio.write_crs(crs_ref, inplace=True)
        try:
            da_3857 = valid.rio.reproject('EPSG:3857')
        except Exception:
            da_3857 = valid
        vmin, vmax = np.nanpercentile(vals, [2, 98])
        if vmax - vmin < 0.1:
            vmin, vmax = vmin - 0.5, vmax + 0.5
        da_3857.plot(ax=ax, cmap='viridis', vmin=vmin, vmax=vmax, add_colorbar=True,
                     cbar_kwargs={'label': 'WSE (m)', 'shrink': 0.7}, alpha=0.8, zorder=2)
        crs_plot = da_3857.rio.crs
        mask_gdf_r = gpd.GeoDataFrame(geometry=geoms, crs=crs_ref)
        station_gdf_r = gpd.GeoDataFrame(geometry=[station_geom], crs=crs_ref)
        plot_water_mask(ax, mask_gdf_r, crs_ref, plot_crs=crs_plot, zorder=3, facecolor='none', edgecolor='red', linewidth=2)
        plot_station_marker(ax, station_gdf_r, crs_ref, plot_crs=crs_plot, zorder=4)
        add_satellite_basemap(ax, crs=crs_plot)
        ax.set_title(f'Station {station_code}\nWSE: {wse_median:.2f} m | px: {n_pixels}', fontsize=12)
        ax.set_axis_off()
        save_figure_safely(fig, out_path)
    except Exception:
        plt.close('all')


def process_raster_granule(nc_path, temp_dir, mask_pickle, station_pickle, maps_dir, station_code, generate_map=True):
    """Clips one Raster granule to the station's water mask, applies the
    quality/water-fraction filters, and returns a dict with the median WSE
    and ancillary statistics — or None if the granule doesn't overlap, has no
    valid pixels after filtering, or fails to open/convert."""
    stem = nc_path.stem
    tif_path = Path(temp_dir) / f'{stem}.tif'
    if not nc_path.exists() or nc_path.stat().st_size < 1000:
        return None
    if not _nc_to_tif(nc_path, tif_path, variable='wse'):
        return None

    result = None
    try:
        mask_gdf = gpd.read_file(mask_pickle)
        station_gdf = gpd.read_file(station_pickle)
        geoms = list(mask_gdf.geometry)
        station_geom = station_gdf.geometry.iloc[0]
        vector_crs = mask_gdf.crs

        with rioxarray.open_rasterio(tif_path) as ref_tif:
            raster_crs = ref_tif.rio.crs
            if not raster_crs:
                return None
            with xr.open_dataset(nc_path, engine='h5netcdf', decode_coords='all', decode_times=False) as ds:
                try:
                    ds = ds.assign_coords(x=ref_tif.x, y=ref_tif.y)
                    ds.rio.write_crs(raster_crs, inplace=True)
                except Exception:
                    return None

                t = Transformer.from_crs(vector_crs, raster_crs, always_xy=True)
                geoms_r = [shp_transform(lambda x, y: t.transform(x, y), g) for g in geoms]
                station_r = shp_transform(lambda x, y: t.transform(x, y), station_geom)

                r_xmin, r_ymin, r_xmax, r_ymax = ds.x.min().item(), ds.y.min().item(), ds.x.max().item(), ds.y.max().item()
                g_xmin, g_ymin, g_xmax, g_ymax = geoms_r[0].bounds
                margin = 1000
                if (r_xmin > g_xmax + margin) or (r_xmax < g_xmin - margin) or (r_ymin > g_ymax + margin) or (r_ymax < g_ymin - margin):
                    return None

                ds_clip = ds.rio.clip(geoms_r, all_touched=True)
                if 'wse' not in ds_clip:
                    return None

                # Quality-flag filters (ordinal, 0=good ... 3=bad; keep < QUALITY_FLAG_MAX)
                for qual_col in ('wse_qual', 'water_area_qual', 'wse_area_qual'):
                    if qual_col in ds_clip:
                        ds_clip = ds_clip.where(ds_clip[qual_col] < QUALITY_FLAG_MAX)

                if 'water_frac' in ds_clip:
                    ds_clip = ds_clip.where(ds_clip['water_frac'] > WATER_FRAC_MIN)

                ds_clip = ds_clip.where((ds_clip['wse'] > -100) & (ds_clip['wse'] < 10000))

                # Locate the ONE cell whose WSE is the median value (exact
                # rank, no interpolation -- always a real observed cell), and
                # report every other attribute from that SAME cell, the same
                # way Node/Reach report one feature's own attributes instead
                # of an aggregate across many.
                wse_flat = ds_clip['wse'].values.flatten()
                valid_positions = np.where(~np.isnan(wse_flat))[0]
                wse_clean = wse_flat[valid_positions]
                if len(wse_clean) > 0:
                    order = np.argsort(wse_clean)
                    median_local_idx = order[(len(wse_clean) - 1) // 2]
                    median_idx_flat = int(valid_positions[median_local_idx])
                    wse_median = float(wse_flat[median_idx_flat])
                    result = {
                        'file': stem, 'date': extract_datetime_from_filename(stem), 'wse_median': round(wse_median, 4),
                        'wse_std': round(float(np.std(wse_clean)), 4), 'n_pixels': len(wse_clean), 'crs': str(raster_crs),
                    }
                    for var in RASTER_AUX_VARS:
                        result[f'{var}_median'] = _raster_value_at_median_cell(ds_clip, var, median_idx_flat)
                    if generate_map:
                        _render_raster_map(ds_clip['wse'], raster_crs, geoms_r, station_r,
                                            maps_dir / f'map_{stem}.png', station_code, wse_median, len(wse_clean))
    except Exception:
        result = None
    finally:
        if tif_path.exists():
            try:
                tif_path.unlink()
            except Exception:
                pass
    return result


def _raster_search(bbox, patterns=None):
    results, seen = [], set()
    try:
        items = earthaccess.search_data(short_name=SHORT_NAME_RASTER, bounding_box=bbox, temporal=(DATE_START, DATE_END),
                                         granule_name='*_100m_*', count=1000)
    except Exception:
        return results
    for item in items:
        granule_ur = item['umm']['GranuleUR']
        pattern = _raster_extract_tile_pattern(granule_ur)
        if patterns and pattern not in patterns:
            continue
        timestamp = extract_datetime_from_filename(granule_ur) or granule_ur
        key = (pattern or granule_ur, timestamp)
        if key not in seen:
            results.append(item)
            seen.add(key)
    return results


def discover_raster_tiles(station_code, bbox, geoms, crs, folder):
    """One-off scan (result cached to disk) of which Raster tile patterns
    actually overlap this station's water mask."""
    temp_dir = folder / 'raster_temp'
    temp_dir.mkdir(exist_ok=True)
    candidates = _raster_search(bbox)
    if not candidates:
        return set()
    grouped = {}
    for item in candidates:
        pattern = _raster_extract_tile_pattern(item['umm']['GranuleUR'])
        if pattern:
            grouped.setdefault(pattern, []).append(item)

    valid = set()
    for pattern, items in grouped.items():
        test_item = items[-1]
        try:
            files = download_with_retries([test_item], temp_dir)
            nc_file = next((Path(f) for f in files if str(f).endswith('.nc')), None)
            if not nc_file or nc_file.stat().st_size <= 1000:
                continue
            tif_file = temp_dir / f'test_{pattern}.tif'
            if _nc_to_tif(nc_file, tif_file):
                with rioxarray.open_rasterio(tif_file) as da:
                    if da.rio.crs:
                        t = Transformer.from_crs(crs, da.rio.crs, always_xy=True)
                        geoms_r = [shp_transform(lambda x, y: t.transform(x, y), g) for g in geoms]
                        try:
                            if da.rio.clip(geoms_r, all_touched=True).count() > 0:
                                valid.add(pattern)
                        except Exception:
                            pass
            if tif_file.exists():
                tif_file.unlink()
            if nc_file.exists():
                nc_file.unlink()
        except Exception:
            pass

    (folder / 'raster_valid_tiles.json').write_text(json.dumps({'patterns': sorted(valid)}))
    return valid


def process_station_raster(station_code, bbox, water_mask_gdf, station_gdf):
    folder = station_folder(station_code)
    temp_dir = folder / 'raster_temp'
    maps_dir = folder / 'raster_maps'
    csv_path = folder / 'raster.csv'
    history_path = folder / 'raster_attempted.txt'
    mask_pickle = folder / 'raster_mask.gpkg'
    station_pickle = folder / 'raster_station.gpkg'
    water_mask_gdf.to_file(mask_pickle, driver='GPKG')
    station_gdf.to_file(station_pickle, driver='GPKG')
    crs = water_mask_gdf.crs
    geoms = list(water_mask_gdf.geometry)

    cleanup_stale_extract_dirs(temp_dir)  # also clears stray .lock/.tif/.xml leftovers below
    for pattern in ('*.lock', '*.tif', '*.xml', '*.aux'):
        for f in temp_dir.glob(pattern) if temp_dir.exists() else []:
            try:
                f.unlink()
            except Exception:
                pass

    tiles_file = folder / 'raster_valid_tiles.json'
    if tiles_file.exists():
        patterns = set(json.loads(tiles_file.read_text()).get('patterns', []))
    else:
        patterns = discover_raster_tiles(station_code, bbox, geoms, crs, folder)
    if not patterns:
        log(f'[{station_code}][Raster] no overlapping tiles found.')
        return

    granules = _raster_search(bbox, patterns)

    attempted = set()
    if csv_path.exists():
        try:
            attempted.update(pd.read_csv(csv_path, usecols=['file'], engine='c')['file'].astype(str).unique())
        except Exception:
            pass
    if history_path.exists():
        attempted.update(line.strip() for line in history_path.read_text().splitlines() if line.strip())

    existing_rows = read_csv_row_count(csv_path) or 0
    to_process = [g for g in granules if Path(g['umm']['GranuleUR']).stem not in attempted]
    if not to_process:
        log(f'[{station_code}][Raster] up to date, nothing new ({existing_rows} row(s) already saved).')
        try:
            mask_pickle.unlink()
            station_pickle.unlink()
        except Exception:
            pass
        return
    log(f'[{station_code}][Raster] {existing_rows} row(s) already saved; {len(to_process)} new candidate(s) '
        f'(of {len(granules)} found for the relevant tiles).')

    def _mark_attempted(batch):
        with open(history_path, 'a') as f:
            for g in batch:
                f.write(f"{Path(g['umm']['GranuleUR']).stem}\n")

    saved = 0

    def _on_result(nc_path, res):
        nonlocal saved
        if res:
            saved += append_rows_aligned(csv_path, res, dedup_key=PRODUCT_SPEC['raster']['key'])

    run_batched_download_pipeline(
        to_process, temp_dir, batch_size=RASTER_DOWNLOAD_CHUNK, n_workers=N_FILE_THREADS,
        process_fn=lambda p: process_raster_granule(p, temp_dir, mask_pickle, station_pickle, maps_dir, station_code),
        on_result=_on_result, on_batch_start=_mark_attempted, label=f'[{station_code}][Raster]',
    )
    cleanup_stale_extract_dirs(temp_dir)

    try:
        mask_pickle.unlink()
        station_pickle.unlink()
    except Exception:
        pass
    log(f'[{station_code}][Raster] done: +{saved} new row(s). Total: {read_csv_row_count(csv_path) or 0} row(s).')


def drop_low_pixel_count_rows(station_codes=None, min_pixels=RASTER_MIN_VALID_PIXELS):
    """A Raster overpass whose water-mask clip contains only 1-2 valid pixels
    produces a median that is really just noise. This removes such rows
    (backing up the CSV first) without touching the checkpoint — the granule
    stays marked as processed, it just won't be re-attempted or re-added."""
    if station_codes is None:
        station_codes = [d.name.replace('Station_', '') for d in sorted(OUTPUT_DIR.glob('Station_*'))]
    for code in station_codes:
        folder = station_folder(code)
        csv_path = folder / 'raster.csv'
        maps_dir = folder / 'raster_maps'
        if not csv_path.exists():
            continue
        df = pd.read_csv(csv_path, engine='python', on_bad_lines='skip')
        if 'n_pixels' not in df.columns:
            continue
        n_col = pd.to_numeric(df['n_pixels'], errors='coerce')
        bad = n_col.notna() & (n_col <= min_pixels)
        if not bad.any():
            continue
        removed = df[bad]
        backup = csv_path.with_suffix('.csv.bak')
        if not backup.exists():
            shutil.copy2(csv_path, backup)
        df[~bad].to_csv(csv_path, index=False)
        if maps_dir.exists():
            for stem in removed.get('file', pd.Series(dtype=str)).dropna().astype(str):
                map_path = maps_dir / f'map_{stem}.png'
                if map_path.exists():
                    map_path.unlink()
        log(f'[{code}][Raster] removed {int(bad.sum())} row(s) with <= {min_pixels} valid pixel(s).')


In [ ]:
# =============================================================================
# BLOCK 4 — NODE / REACH EXTRACTION (RiverSP vector product)
# =============================================================================
# Node and Reach share the exact same matching logic (find the SWORD
# node/reach polyline closest to the station, inside the water mask, within
# MAX_MATCH_DISTANCE_KM) and differ only in which shapefile inside the RiverSP
# ZIP they read — so a single worker handles both, selected by `feature_type`.


def process_river_granule(args):
    """Downloads one RiverSP granule ZIP, extracts the Node or Reach shapefile
    it contains, and keeps the single feature (if any) that both intersects
    the station's water mask and lies within MAX_MATCH_DISTANCE_KM of the
    station point. Returns (rows, technical_success, granule_stem, diagnostics).

    `technical_success` is True whenever the granule was opened and evaluated
    at all — including when it produced zero rows (e.g. it simply doesn't
    reach this station) — so it can be checkpointed and never re-downloaded;
    only a genuine download/IO failure returns False, leaving it eligible for
    retry on the next run.
    """
    granule, temp_dir, vector_out_dir, mask_pickle, station_pickle, feature_type = args

    diag = {
        'no_shapefile_of_type': 0, 'empty_shapefile': 0, 'outside_mask_bbox': 0,
        'empty_intersection': 0, 'beyond_max_distance': 0, 'matched_within_distance': 0,
        'shapefile_read_error': 0,
    }
    last_shp_error = None

    try:
        filename = Path(granule).name if isinstance(granule, str) else granule['meta']['native-id']
        if not filename.endswith('.zip'):
            filename += '.zip'
    except Exception:
        return (None, False, 'id_error', diag)

    local_path = Path(temp_dir) / filename
    extract_dir = Path(temp_dir) / f'ext_{os.getpid()}_{threading.get_ident()}_{int(time.time() * 1000)}'
    rows = []
    technical_success = False

    try:
        if not local_path.exists():
            if not robust_download(granule, temp_dir, filename):
                return (None, False, 'download_failed', diag)

        engine = VECTOR_ENGINE
        mask_gdf = gpd.read_file(mask_pickle, engine=engine)
        station_gdf = gpd.read_file(station_pickle, engine=engine)
        extract_dir.mkdir(exist_ok=True)

        try:
            with zipfile.ZipFile(local_path, 'r') as z:
                z.extractall(extract_dir)
        except Exception:
            local_path.unlink()
            return (None, False, 'corrupted_zip', diag)

        shapefiles = list(extract_dir.rglob(f'*{feature_type}*.shp')) or \
            [p for p in extract_dir.rglob('*.shp') if not p.name.startswith('.')]
        if not shapefiles:
            diag['no_shapefile_of_type'] += 1

        # A granule's ZIP can (rarely) contain more than one shapefile of the
        # requested type. We first collect the closest match from EACH
        # shapefile, then keep only the globally closest one across the whole
        # granule — one output row per granule, never more.
        matches = []  # (distance_km, selected_row, feature_gdf, source_shp_name, original_columns)
        for shp in shapefiles:
            try:
                gdf = gpd.read_file(shp, engine=engine)
                if gdf.empty:
                    diag['empty_shapefile'] += 1
                    continue
                # SWORD occasionally ships a shapefile with valid geometry but
                # literally no attribute columns for a given granule. Treat it
                # the same as an empty shapefile: skip it, rather than let it
                # become a "match" with no scientific data, which would poison
                # the CSV's header for every future write (see
                # `sanitize_existing_csv`).
                if len([c for c in gdf.columns if c != 'geometry']) == 0:
                    diag['no_attributes'] = diag.get('no_attributes', 0) + 1
                    continue
                if gdf.crs != mask_gdf.crs:
                    gdf = gdf.to_crs(mask_gdf.crs)

                minx, miny, maxx, maxy = gdf.total_bounds
                if not list(mask_gdf.sindex.intersection((minx, miny, maxx, maxy))):
                    diag['outside_mask_bbox'] += 1
                    continue

                intersection = gpd.overlay(gdf, mask_gdf, how='intersection', keep_geom_type=False)
                if intersection.empty:
                    diag['empty_intersection'] += 1
                    continue

                station_point = station_gdf.geometry.iloc[0]
                calc = intersection.copy()
                if calc.crs.is_geographic:
                    utm = calc.estimate_utm_crs()
                    calc_proj = calc.to_crs(utm)
                    station_proj = gpd.GeoSeries([station_point], crs=station_gdf.crs).to_crs(utm).iloc[0]
                    calc['distance_km'] = calc_proj.geometry.distance(station_proj) / 1000
                else:
                    calc['distance_km'] = calc.geometry.distance(station_point) / 1000

                selected = calc[calc['distance_km'] <= MAX_MATCH_DISTANCE_KM].copy()
                if not selected.empty:
                    diag['matched_within_distance'] += 1
                    selected = selected.sort_values('distance_km').iloc[:1]
                    feature_gdf = intersection.loc[selected.index]
                    matches.append((float(selected['distance_km'].iloc[0]), selected, feature_gdf, shp.name, list(gdf.columns)))
                else:
                    diag['beyond_max_distance'] += 1
            except Exception as shp_exc:
                diag['shapefile_read_error'] += 1
                last_shp_error = f'{type(shp_exc).__name__}: {str(shp_exc)[:150]}'

        if matches:
            matches.sort(key=lambda m: m[0])
            _dist, selected, feature_gdf, _winner_shp_name, _original_cols = matches[0]

            # Debug shapefile/map export is best-effort and isolated: a mixed
            # geometry type from the overlay (LineString + GeometryCollection)
            # can make the ESRI Shapefile driver refuse to write, but that must
            # never cost us the actual WSE data below.
            base_name = f'{local_path.stem}_matched'
            vector_zip_path = Path(vector_out_dir) / f'{base_name}.zip'
            export_ok = False
            try:
                export_gdf = feature_gdf.copy()
                for col in export_gdf.columns:
                    if pd.api.types.is_datetime64_any_dtype(export_gdf[col]):
                        export_gdf[col] = export_gdf[col].astype(str)
                shp_out = extract_dir / f'{base_name}.shp'
                export_gdf.to_file(shp_out)
                with zipfile.ZipFile(vector_zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
                    for sidecar in extract_dir.glob(f'{base_name}.*'):
                        zf.write(sidecar, arcname=sidecar.name)
                export_ok = True
            except Exception:
                pass

            try:
                fig, ax = plt.subplots(figsize=(10, 10))
                feature_gdf.to_crs(epsg=3857).plot(ax=ax, color='red', linewidth=3, zorder=5)
                plot_water_mask(ax, mask_gdf, mask_gdf.crs, zorder=1)
                plot_station_marker(ax, station_gdf, station_gdf.crs, zorder=10)
                add_satellite_basemap(ax, zoom=13)
                ax.set_axis_off()
                ax.set_title(f'{feature_type} - {local_path.stem}', fontsize=10)
                save_figure_safely(fig, Path(vector_out_dir) / f'{local_path.stem}_map.png')
            except Exception:
                plt.close('all')

            for _, row in selected.iterrows():
                d = row.drop('geometry').to_dict()
                d['source_file'] = local_path.name
                d['distance_to_station_km'] = row['distance_km']
                d['matched_shapefile_zip'] = vector_zip_path.name if export_ok else None
                m = re.search(r'_(\d{8}T\d{6})_', local_path.name)
                if m:
                    d['file_timestamp'] = m.group(1)
                m = re.search(r'_\d{3}_(\d{3})_', local_path.name)
                if m:
                    d['pass_id'] = m.group(1)
                rows.append(d)

        technical_success = True

    except Exception as fatal_exc:
        return (None, False, f'worker_error: {type(fatal_exc).__name__}: {str(fatal_exc)[:150]}', diag)
    finally:
        if extract_dir.exists():
            shutil.rmtree(extract_dir, ignore_errors=True)
        try:
            if local_path.exists():
                local_path.unlink()
        except Exception:
            pass

    if diag['shapefile_read_error'] and last_shp_error:
        diag['last_shapefile_error'] = last_shp_error
    return (rows, technical_success, local_path.stem, diag)


def sanitize_existing_csv(csv_path, feature_type, checkpoint=None):
    """Repairs two kinds of corruption an interrupted/concurrent run can leave
    behind, before a fresh run starts writing to the same file:

    1. A "ragged" CSV (rows with different column counts) — read tolerantly,
       back up the original, keep only well-formed rows.
    2. A CSV whose header never got the scientific columns (wse, node_q /
       reach_q) — this happens if the very first batch ever written for this
       station/type came from a shapefile with no SWORD attributes (see
       `process_river_granule`); every later write gets silently realigned to
       that poor header by `append_rows_aligned`, discarding wse forever
       without raising any error. Detected as: non-empty CSV missing 'wse'.
       Treated as unrecoverable — the file is deleted so the next valid batch
       rebuilds its header from scratch — and, if a checkpoint was passed in,
       every granule of this feature_type it had marked "done" is unmarked,
       since their data has no surviving row anywhere.
    """
    path = Path(csv_path)
    if not path.exists():
        return
    corrupted = False
    try:
        df = pd.read_csv(path)
    except Exception:
        corrupted = True
        try:
            df = pd.read_csv(path, on_bad_lines='skip', engine='python')
        except Exception:
            return
        backup = path.with_suffix(path.suffix + '.corrupted.bak')
        if not backup.exists():
            shutil.copy2(path, backup)

    if not corrupted and len(df) > 0 and 'wse' not in df.columns:
        backup = path.with_suffix(path.suffix + '.corrupted.bak')
        if not backup.exists():
            shutil.copy2(path, backup)
        if checkpoint is not None:
            marker = f'_{feature_type}_'
            suspects = [pid for pid in list(checkpoint.done) if marker in pid]
            if suspects:
                checkpoint.unmark(suspects)
        path.unlink()
        return

    if len(df) == 0:
        if corrupted:
            df.to_csv(path, index=False)
        return

    quality_col = 'reach_q' if feature_type == 'Reach' else 'node_q'
    rows_before = len(df)
    if 'wse' in df.columns:
        df['wse'] = pd.to_numeric(df['wse'], errors='coerce')
        df = df[df['wse'] > -1e9]
    if quality_col in df.columns:
        df[quality_col] = pd.to_numeric(df[quality_col], errors='coerce')
        df = df[df[quality_col] < QUALITY_FLAG_MAX]

    if corrupted or len(df) != rows_before:
        df.to_csv(path, index=False)

    if corrupted and checkpoint is not None and 'source_file' in df.columns:
        stems_present = {Path(str(a)).stem for a in df['source_file'].dropna().astype(str)}
        marker = f'_{feature_type}_'
        suspects = [pid for pid in list(checkpoint.done) if marker in pid and pid not in stems_present]
        if suspects:
            checkpoint.unmark(suspects)


def process_station_river(station_code, bbox, mask_gdf_all, station_gdf_all):
    code = str(station_code).replace('.0', '')
    folder = station_folder(code)
    checkpoint = ProcessingCheckpoint(folder)

    local_mask = mask_gdf_all[mask_gdf_all[STATION_CODE_FIELD].astype(str).str.replace('.0', '') == code]
    if local_mask.empty:
        log(f'[{code}][Node/Reach] no water mask found, skipping.')
        return
    local_station = station_gdf_all[station_gdf_all[STATION_CODE_FIELD].astype(str).str.replace('.0', '') == code]
    if local_station.empty:
        local_station = gpd.GeoDataFrame(geometry=[local_mask.geometry.centroid.iloc[0]], crs=local_mask.crs)

    mask_pickle = folder / 'river_mask.gpkg'
    station_pickle = folder / 'river_station.gpkg'
    local_mask.to_file(mask_pickle, driver='GPKG')
    local_station.to_file(station_pickle, driver='GPKG')

    for feature_type in ('Reach', 'Node'):
        temp_dir = folder / f'{feature_type.lower()}_temp'
        vector_dir = folder / f'{feature_type.lower()}_vectors'
        csv_path = folder / f'{feature_type.lower()}.csv'
        temp_dir.mkdir(parents=True, exist_ok=True)
        vector_dir.mkdir(parents=True, exist_ok=True)

        cleanup_stale_extract_dirs(temp_dir)
        sanitize_existing_csv(csv_path, feature_type, checkpoint)

        try:
            granules = earthaccess.search_data(short_name=SHORT_NAME_RIVERSP, bounding_box=bbox,
                                                temporal=(DATE_START, DATE_END), granule_name=f'*_{feature_type}_*') or []
        except Exception:
            granules = []

        tasks = []
        for g in granules:
            stem = Path(g['meta']['native-id']).stem
            if not checkpoint.is_done(stem):
                tasks.append((g, str(temp_dir), str(vector_dir), str(mask_pickle), str(station_pickle), feature_type))
        existing_rows = read_csv_row_count(csv_path) or 0
        if not tasks:
            log(f'[{code}][{feature_type}] up to date, nothing new ({existing_rows} row(s) already saved).')
            continue

        log(f'[{code}][{feature_type}] {existing_rows} row(s) already saved; processing {len(tasks)} new '
            f'candidate(s) with {max(1, min(N_FILE_THREADS, len(tasks)))} threads.')
        quality_col = 'reach_q' if feature_type == 'Reach' else 'node_q'
        csv_lock = threading.Lock()
        done_ids, n_saved, n_technical_ok = [], 0, 0

        def _filter_and_save(records):
            nonlocal n_saved
            df_new = pd.DataFrame(records)
            if 'wse' in df_new.columns:
                df_new['wse'] = pd.to_numeric(df_new['wse'], errors='coerce')
                df_new = df_new[df_new['wse'] > -1e9]
            if quality_col in df_new.columns:
                df_new[quality_col] = pd.to_numeric(df_new[quality_col], errors='coerce')
                df_new = df_new[df_new[quality_col] < QUALITY_FLAG_MAX]
            if df_new.empty:
                return
            n_saved += append_rows_aligned(csv_path, df_new, lock=csv_lock, dedup_key=PRODUCT_SPEC[feature_type.lower()]['key'])

        with ThreadPoolExecutor(max_workers=max(1, min(N_FILE_THREADS, len(tasks)))) as executor:
            futures = {executor.submit(process_river_granule, task): task for task in tasks}
            count = 0
            for future in as_completed(futures):
                count += 1
                try:
                    rows, success, granule_id, _diag = future.result()
                except Exception:
                    rows, success, granule_id = None, False, None

                if success:
                    done_ids.append(granule_id)
                    n_technical_ok += 1
                    if rows:
                        _filter_and_save(rows)

                print_progress(f'[{code}][{feature_type}]', count, len(tasks), f'saved so far: {n_saved}')
                if count % 16 == 0 or count == len(tasks):
                    cleanup_stale_extract_dirs(temp_dir, only_old=True)
                    gc.collect()
                if len(done_ids) >= 20 or count == len(tasks):
                    if done_ids:
                        checkpoint.mark_done(done_ids)
                        done_ids = []

        if n_saved > 0:
            deduplicate_csv(csv_path, subset=['file_timestamp', 'pass_id'])
        log(f'[{code}][{feature_type}] done: +{n_saved} new row(s), {n_technical_ok} granule(s) technically '
            f'completed. Total: {read_csv_row_count(csv_path) or 0} row(s).')

    try:
        mask_pickle.unlink()
        station_pickle.unlink()
    except Exception:
        pass


In [ ]:
# =============================================================================
# BLOCK 5 — REACH WSE POSITION CORRECTION
# =============================================================================
# The Reach product reports WSE fitted at the *center* of the ~10 km reach
# (`p_dist_out`, the along-river distance from the reach center to the
# network outlet — JPL D-56413 Product Description, Sec. 4.1.11), not at the
# gauge location. We translate it to the gauge position along the river
# centerline using:
#
#     wse_corrected = wse + delta_x * slope
#
# where:
#   slope   = Reach water-surface slope (m/m). Per the product definition
#             (JPL D-56413, Sec. 4.1.5): "a positive slope means the
#             downstream WSE is lower" — i.e. WSE increases in the direction
#             of increasing p_dist_out (upstream).
#   delta_x = p_dist_out(closest Node to the gauge) - p_dist_out(reach center),
#             in meters. Node also reports p_dist_out (distance from that
#             specific node to the outlet), so this is the along-centerline
#             distance between the gauge and the reach center, expressed in
#             the same reference the slope sign convention assumes.
#
# The "point measured along the river centerline closest to the gauge" is
# found among that reach's Node records, by minimizing the great-circle
# distance between each node's observed (lat, lon) and the gauge coordinates.
#
# Where no Node record is available for a reach (e.g. Node extraction found
# nothing for that station), delta_x cannot be computed and the row falls
# back to the raw (uncorrected) wse — so no row/station is silently dropped
# from downstream analysis for lack of a correction.

import math


def _haversine_m(lat1, lon1, lat2, lon2):
    R = 6371000.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    d_phi = math.radians(lat2 - lat1)
    d_lambda = math.radians(lon2 - lon1)
    a = math.sin(d_phi / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(d_lambda / 2) ** 2
    return 2 * R * math.asin(math.sqrt(a))


def _nearest_node_dist_out_by_reach(node_df, gauge_lat, gauge_lon):
    required = {'reach_id', 'node_id', 'lat', 'lon', 'p_dist_out'}
    if node_df is None or node_df.empty or not required.issubset(node_df.columns):
        return {}
    nodes = node_df.dropna(subset=['reach_id', 'lat', 'lon', 'p_dist_out']).drop_duplicates(subset=['reach_id', 'node_id'])
    result = {}
    for reach_id, group in nodes.groupby('reach_id'):
        distances = group.apply(lambda r: _haversine_m(gauge_lat, gauge_lon, r['lat'], r['lon']), axis=1)
        result[reach_id] = float(group.loc[distances.idxmin(), 'p_dist_out'])
    return result


def compute_wse_corrected(station_codes, stations_gdf, slope_column='slope'):
    """For every station in `station_codes`, adds `wse_corrected` (and the
    diagnostic `delta_x_to_gauge_m`) to that station's reach.csv, writing the
    result back to disk (with a .bak backup of the previous version)."""
    stations_wgs84 = stations_gdf.to_crs('EPSG:4326')
    summary = []

    for code in station_codes:
        code = str(code).replace('.0', '')
        folder = station_folder(code)
        reach_csv = folder / 'reach.csv'
        node_csv = folder / 'node.csv'
        if not reach_csv.exists():
            continue

        reach_df = pd.read_csv(reach_csv, low_memory=False)
        required_cols = {'reach_id', 'p_dist_out', slope_column, 'wse'}
        if not required_cols.issubset(reach_df.columns):
            summary.append({'station': code, 'rows': len(reach_df), 'corrected': 0, 'reason': 'reach.csv missing required columns'})
            continue

        gauge_rows = stations_wgs84[stations_wgs84[STATION_CODE_FIELD].astype(str).str.replace('.0', '') == code]
        if gauge_rows.empty:
            summary.append({'station': code, 'rows': len(reach_df), 'corrected': 0, 'reason': 'station not found in stations layer'})
            continue
        gauge_point = gauge_rows.geometry.iloc[0]
        gauge_lat, gauge_lon = gauge_point.y, gauge_point.x

        node_df = pd.read_csv(node_csv, low_memory=False) if node_csv.exists() else pd.DataFrame()
        dist_out_at_gauge = _nearest_node_dist_out_by_reach(node_df, gauge_lat, gauge_lon)

        gauge_dist_out = reach_df['reach_id'].map(dist_out_at_gauge)
        reach_center_dist_out = pd.to_numeric(reach_df['p_dist_out'], errors='coerce')
        slope = pd.to_numeric(reach_df[slope_column], errors='coerce')
        wse = pd.to_numeric(reach_df['wse'], errors='coerce')

        delta_x = gauge_dist_out - reach_center_dist_out
        corrected = wse + delta_x * slope
        reach_df['delta_x_to_gauge_m'] = delta_x
        # Fall back to the raw wse wherever no correction could be computed,
        # so a station is never dropped from downstream analysis purely for
        # lack of a Node reference — see module docstring above.
        reach_df['wse_corrected'] = corrected.where(corrected.notna(), wse)

        n_corrected = int(delta_x.notna().sum())
        backup = reach_csv.with_suffix('.csv.bak')
        if not backup.exists():
            shutil.copy2(reach_csv, backup)
        reach_df.to_csv(reach_csv, index=False)
        summary.append({'station': code, 'rows': len(reach_df), 'corrected': n_corrected,
                         'reason': 'ok' if n_corrected else 'no matching Node reference for this reach'})

    summary_df = pd.DataFrame(summary)
    log('=' * 80)
    log('REACH WSE POSITION CORRECTION')
    log('=' * 80)
    if not summary_df.empty:
        log(summary_df.to_string(index=False))
    return summary_df


In [ ]:
# =============================================================================
# BLOCK 6 — CROSS-PRODUCT COMPLETENESS CHECK AND RECOVERY
# =============================================================================
# For every station, compares which overpass dates produced valid data in
# each of the four products. A date with valid data in at least one product
# but missing from another is re-downloaded and re-run through that product's
# real extraction worker (the same functions used in Blocks 2-4 above, not a
# lighter re-implementation) to determine, conclusively, whether the gap is a
# genuine outcome (outside the water mask, quality-filtered, no SWORD match,
# ...) or a recoverable one — in which case the row is added to the CSV.
#
# Every date ever checked is cached per station/product
# (`{product}_dates_checked.json`), so re-running this block never repeats
# work — a transient failure (network/download error) is the only kind of
# result NOT cached, so it is retried automatically on the next run.

def _dates_in_csv(csv_path, column, fmt):
    p = Path(csv_path)
    if not p.exists():
        return set()
    try:
        df = pd.read_csv(p, usecols=[column], engine='python', on_bad_lines='skip')
    except Exception:
        return set()
    days = set()
    for v in df[column].dropna().astype(str):
        if fmt == 'iso':
            days.add(v[:10])
        elif fmt == 'compact' and len(v) >= 8 and v[:8].isdigit():
            days.add(f'{v[:4]}-{v[4:6]}-{v[6:8]}')
    return days


def _check_date_pixc(station_code, date, ctx):
    d0 = datetime.strptime(date, '%Y-%m-%d')
    temporal = (d0.strftime('%Y-%m-%d'), (d0 + pd.Timedelta(days=1)).strftime('%Y-%m-%d'))
    try:
        candidates = earthaccess.search_data(short_name=SHORT_NAME_PIXC, bounding_box=ctx['bbox'], temporal=temporal)
    except Exception as e:
        return [{'date': date, 'status': f'error_cmr_search: {type(e).__name__}'}], 0, True
    if not candidates:
        return [{'date': date, 'status': 'no_pixc_granule_found'}], 0, False

    details, recovered, transient = [], 0, False
    for g in candidates:
        files = download_with_retries([g], ctx['temp_dir'])
        nc_file = next((Path(f) for f in files if str(f).endswith('.nc')), None)
        if not nc_file:
            details.append({'date': date, 'status': 'failed_download'})
            transient = True
            continue
        res = process_pixc_granule(nc_file, ctx['water_mask_gdf'], ctx['station_gdf'], ctx['maps_dir'], station_code, generate_map=False)
        try:
            nc_file.unlink()
        except Exception:
            pass
        status = res.get('status', 'unknown')
        if status == 'ok':
            row = {k: v for k, v in res.items() if k != 'status'}
            append_rows_aligned(ctx['csv_path'], row, lock=ctx['csv_lock'], dedup_key=PRODUCT_SPEC['pixc']['key'])
            recovered += 1
            status = 'recovered'
        details.append({'date': date, 'status': status})
        if is_transient_status(status):
            transient = True
    return details, recovered, transient


def _check_date_raster(station_code, date, ctx):
    d0 = datetime.strptime(date, '%Y-%m-%d')
    temporal = (d0.strftime('%Y-%m-%d'), (d0 + pd.Timedelta(days=1)).strftime('%Y-%m-%d'))
    try:
        candidates = earthaccess.search_data(short_name=SHORT_NAME_RASTER, bounding_box=ctx['bbox'], temporal=temporal, granule_name='*_100m_*')
    except Exception as e:
        return [{'date': date, 'status': f'error_cmr_search: {type(e).__name__}'}], 0, True
    if not candidates:
        return [{'date': date, 'status': 'no_raster_granule_found'}], 0, False

    details, recovered, transient = [], 0, False
    for g in candidates:
        files = download_with_retries([g], ctx['temp_dir'])
        nc_file = next((Path(f) for f in files if str(f).endswith('.nc')), None)
        if not nc_file or nc_file.stat().st_size <= 1000:
            details.append({'date': date, 'status': 'failed_download'})
            transient = True
            continue
        res = process_raster_granule(nc_file, ctx['temp_dir'], ctx['mask_pickle'], ctx['station_pickle'],
                                      ctx['maps_dir'], station_code, generate_map=False)
        try:
            nc_file.unlink()
        except Exception:
            pass
        if res is None:
            details.append({'date': date, 'status': 'no_valid_pixels_after_clip_and_filters'})
            continue
        if res.get('n_pixels', 0) <= RASTER_MIN_VALID_PIXELS:
            details.append({'date': date, 'status': 'too_few_pixels'})
            continue
        append_rows_aligned(ctx['csv_path'], res, lock=ctx['csv_lock'], dedup_key=PRODUCT_SPEC['raster']['key'])
        recovered += 1
        details.append({'date': date, 'status': 'recovered'})
    return details, recovered, transient


def _check_date_river(station_code, date, feature_type, ctx):
    d0 = datetime.strptime(date, '%Y-%m-%d')
    temporal = (d0.strftime('%Y-%m-%d'), (d0 + pd.Timedelta(days=1)).strftime('%Y-%m-%d'))
    quality_col = 'reach_q' if feature_type == 'Reach' else 'node_q'
    try:
        candidates = earthaccess.search_data(short_name=SHORT_NAME_RIVERSP, bounding_box=ctx['bbox'], temporal=temporal,
                                              granule_name=f'*_{feature_type}_*', count=10)
    except Exception as e:
        return [{'date': date, 'status': f'error_cmr_search: {type(e).__name__}'}], 0, True
    if not candidates:
        return [{'date': date, 'status': f'no_{feature_type.lower()}_granule_found'}], 0, False

    details, recovered, transient = [], 0, False
    for g in candidates:
        args = (g, str(ctx['temp_dir']), str(ctx['vector_dir']), str(ctx['mask_pickle']), str(ctx['station_pickle']), feature_type)
        try:
            rows, success, granule_id, diag = process_river_granule(args)
        except Exception as e:
            details.append({'date': date, 'status': f'error_processing: {type(e).__name__}'})
            transient = True
            continue

        if not success:
            details.append({'date': date, 'status': f'failed_technical: {granule_id}'})
            transient = True
            continue
        if granule_id:
            with ctx['checkpoint_lock']:
                ctx['checkpoint'].mark_done([granule_id])
        if not rows:
            reason = next((k for k in ('matched_within_distance', 'beyond_max_distance', 'empty_intersection',
                                        'outside_mask_bbox', 'empty_shapefile', 'no_shapefile_of_type') if diag.get(k)), None)
            details.append({'date': date, 'status': reason or 'no_data_undetermined_reason'})
            continue

        row = rows[0]
        if 'wse' not in row:
            details.append({'date': date, 'status': 'matched_shapefile_no_attributes'})
            continue
        try:
            good_quality = float(row.get(quality_col)) < QUALITY_FLAG_MAX
        except (TypeError, ValueError):
            good_quality = False
        if not good_quality:
            details.append({'date': date, 'status': 'quality_filtered'})
            continue

        wse_val = pd.to_numeric(row.get('wse'), errors='coerce')
        if pd.isna(wse_val) or wse_val <= -1e9:
            details.append({'date': date, 'status': 'invalid_wse'})
            continue
        append_rows_aligned(ctx['csv_path'], row, lock=ctx['csv_lock'], dedup_key=PRODUCT_SPEC[feature_type.lower()]['key'])
        recovered += 1
        details.append({'date': date, 'status': 'recovered'})
    return details, recovered, transient


_DATE_CHECKERS = {
    'pixc': _check_date_pixc,
    'raster': _check_date_raster,
    'node': lambda code, date, ctx: _check_date_river(code, date, 'Node', ctx),
    'reach': lambda code, date, ctx: _check_date_river(code, date, 'Reach', ctx),
}


def _build_station_context(code, mask_gdf_all, station_gdf_all, product):
    folder = station_folder(code)
    sel = mask_gdf_all[STATION_CODE_FIELD].astype(str).str.replace('.0', '') == code
    rows_mask = mask_gdf_all[sel]
    if rows_mask.empty:
        return None
    rows_station = station_gdf_all[station_gdf_all[STATION_CODE_FIELD].astype(str).str.replace('.0', '') == code]
    if rows_station.empty:
        rows_station = gpd.GeoDataFrame(geometry=[rows_mask.geometry.centroid.iloc[0]], crs=rows_mask.crs)

    bbox = tuple(rows_mask.to_crs('EPSG:4326').total_bounds)
    temp_dir = folder / f'{product}_recovery_temp'
    temp_dir.mkdir(exist_ok=True)
    ctx = {
        'bbox': bbox, 'temp_dir': temp_dir,
        'csv_path': folder / PRODUCT_SPEC[product]['csv'],
        'csv_lock': threading.Lock(),
    }
    if product in ('pixc', 'raster'):
        ctx['maps_dir'] = folder / f'{product}_maps'
        ctx['maps_dir'].mkdir(exist_ok=True)
        ctx['water_mask_gdf'] = gpd.GeoDataFrame(geometry=list(rows_mask.geometry), crs=rows_mask.crs)
        ctx['station_gdf'] = gpd.GeoDataFrame(geometry=[rows_station.geometry.iloc[0]], crs=rows_mask.crs)
        if product == 'raster':
            ctx['mask_pickle'] = folder / f'{product}_recovery_mask.gpkg'
            ctx['station_pickle'] = folder / f'{product}_recovery_station.gpkg'
            ctx['water_mask_gdf'].to_file(ctx['mask_pickle'], driver='GPKG')
            ctx['station_gdf'].to_file(ctx['station_pickle'], driver='GPKG')
    else:
        ctx['vector_dir'] = folder / f'{product}_vectors'
        ctx['vector_dir'].mkdir(exist_ok=True)
        ctx['mask_pickle'] = folder / f'{product}_recovery_mask.gpkg'
        ctx['station_pickle'] = folder / f'{product}_recovery_station.gpkg'
        rows_mask.to_file(ctx['mask_pickle'], driver='GPKG')
        rows_station.to_file(ctx['station_pickle'], driver='GPKG')
        ctx['checkpoint'] = ProcessingCheckpoint(folder)
        ctx['checkpoint_lock'] = threading.Lock()
    return ctx


def run_completeness_check(product, station_codes, mask_gdf_all, station_gdf_all,
                            max_dates_per_station=None, max_dates_total=100, max_workers=6, force_recheck=False):
    """Runs the cross-product completeness check/recovery for a single
    `product` ('pixc', 'raster', 'node', or 'reach') across `station_codes`.
    Budgeted by `max_dates_total` per call so a full backlog can be worked
    through in safe, resumable batches."""
    login_earthaccess()
    spec = PRODUCT_SPEC[product]
    other_products = [p for p in PRODUCT_SPEC if p != product]

    missing_by_station, cache_by_station, rows_summary = {}, {}, []
    skipped_from_cache = 0

    for code in station_codes:
        code = str(code).replace('.0', '')
        folder = station_folder(code)
        if not folder.exists():
            continue
        dates_by_product = {
            p: _dates_in_csv(folder / PRODUCT_SPEC[p]['csv'], PRODUCT_SPEC[p]['date_column'], PRODUCT_SPEC[p]['date_format'])
            for p in PRODUCT_SPEC
        }
        dates_elsewhere = set()
        for p in other_products:
            dates_elsewhere |= dates_by_product[p]
        missing = sorted(dates_elsewhere - dates_by_product[product])

        cache = {} if force_recheck else load_verified_dates(folder, product)
        cache_by_station[code] = cache
        skipped_from_cache += sum(1 for d in missing if d in cache)
        pending = [d for d in missing if d not in cache]
        if max_dates_per_station is not None:
            pending = pending[:max_dates_per_station]
        missing_by_station[code] = pending
        rows_summary.append({'station': code, f'n_{product}': len(dates_by_product[product]), 'n_pending_check': len(pending)})

    total_pending = sum(len(v) for v in missing_by_station.values())
    log(f'[{product}] dates pending verification: {total_pending} (+ {skipped_from_cache} already checked before, skipped)')

    tasks = [(code, d) for code, dates in missing_by_station.items() for d in dates]
    if max_dates_total is not None and len(tasks) > max_dates_total:
        log(f'[{product}] {len(tasks)} dates pending in total; processing the first {max_dates_total} this call (re-run to continue).')
        tasks = tasks[:max_dates_total]

    contexts = {}
    for code in sorted({code for code, _ in tasks}):
        ctx = _build_station_context(code, mask_gdf_all, station_gdf_all, product)
        if ctx is not None:
            ctx['cache_changed'] = False
            contexts[code] = ctx

    checker = _DATE_CHECKERS[product]
    all_details, total_recovered, done = [], 0, 0
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(checker, code, date, contexts[code]): (code, date) for code, date in tasks if code in contexts}
        for future in as_completed(futures):
            code, date = futures[future]
            done += 1
            try:
                details, recovered, transient = future.result()
            except Exception as e:
                details, recovered, transient = [{'date': date, 'status': f'error_unexpected: {type(e).__name__}'}], 0, True
            all_details.extend({'station': code, **d} for d in details)
            total_recovered += recovered
            if not transient:
                cache_by_station[code][date] = [d.get('status') for d in details]
                contexts[code]['cache_changed'] = True
            print_progress(f'[{product}] progress', done, len(futures), f'recovered so far: {total_recovered}')

    for code, ctx in contexts.items():
        if ctx['cache_changed']:
            save_verified_dates(station_folder(code), product, cache_by_station[code])
        shutil.rmtree(ctx['temp_dir'], ignore_errors=True)
        for key in ('mask_pickle', 'station_pickle'):
            p = ctx.get(key)
            if p and Path(p).exists():
                try:
                    Path(p).unlink()
                except Exception:
                    pass

    if total_recovered:
        for code in {c for c, _ in tasks}:
            deduplicate_csv(station_folder(code) / spec['csv'])

    summary_df = pd.DataFrame(rows_summary)
    detail_df = pd.DataFrame(all_details)
    log(f'[{product}] TOTAL recovered: {total_recovered} new row(s). Dates processed this call: {len(tasks)}.')
    if not detail_df.empty:
        log(detail_df['status'].value_counts().to_string())
    return summary_df, detail_df


In [ ]:
# =============================================================================
# BLOCK 7 — FINAL DUPLICATE AUDIT
# =============================================================================
# `append_rows_aligned` (Block 1) already refuses to write a row whose
# PRODUCT_SPEC key already exists on disk, so a normal run should never
# produce a duplicate in the first place. This is the independent audit that
# verifies that guarantee held — e.g. after importing data from an older run
# that predates the write-time check, or from an external source — rather
# than the primary defense against duplicates.
#
# Where duplicates are found anyway, the row with the fewest missing values
# per key is kept (a reasonable, low-risk tie-breaker: duplicate keys are
# expected to be near-identical re-extractions of the same physical overpass,
# and the more complete one is preferred when they are not).


def run_duplicate_audit(station_codes=None, execute_cleanup=True):
    if station_codes is None:
        station_codes = [d.name.replace('Station_', '') for d in sorted(OUTPUT_DIR.glob('Station_*'))]

    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    rows_summary = []

    for code in station_codes:
        code = str(code).replace('.0', '')
        folder = station_folder(code)
        for product, spec in PRODUCT_SPEC.items():
            key = spec['key']
            csv_path = folder / spec['csv']
            if not csv_path.exists():
                continue
            try:
                df = pd.read_csv(csv_path, low_memory=False)
            except Exception as e:
                rows_summary.append({'station': code, 'product': product, 'error': str(e)})
                continue
            key_cols = [c for c in key if c in df.columns]
            if not key_cols:
                continue

            total = len(df)
            counts = df.groupby(key_cols).size()
            duplicated_keys = counts[counts > 1]
            excess_rows = int((duplicated_keys - 1).sum()) if len(duplicated_keys) else 0

            rows_summary.append({
                'station': code, 'product': product, 'total_rows': total,
                'unique_keys': int(df.drop_duplicates(subset=key_cols).shape[0]),
                'duplicated_keys': int(len(duplicated_keys)), 'excess_rows': excess_rows,
            })

            if excess_rows and execute_cleanup:
                df['_n_missing'] = df.isna().sum(axis=1)
                cleaned = (df.sort_values('_n_missing', kind='mergesort')
                             .drop_duplicates(subset=key_cols, keep='first')
                             .drop(columns='_n_missing'))
                backup = csv_path.with_name(f'{csv_path.name}.bak_{timestamp}')
                shutil.copy2(csv_path, backup)
                cleaned.to_csv(csv_path, index=False)

    summary_df = pd.DataFrame(rows_summary)
    log('=' * 100)
    log('DUPLICATE AUDIT — all stations, all products')
    log('=' * 100)
    with_dups = summary_df[summary_df.get('excess_rows', 0) > 0] if not summary_df.empty else summary_df
    if len(with_dups):
        log(with_dups.sort_values('excess_rows', ascending=False).to_string(index=False))
    else:
        log('No duplicate keys found in any station/product.')
    total_excess = int(summary_df.get('excess_rows', pd.Series(dtype=float)).fillna(0).sum())
    log(f'\nTotal excess (duplicated) rows found: {total_excess}'
        + (f' — removed (backups saved as .bak_{timestamp})' if execute_cleanup and total_excess else ''))
    return summary_df


In [ ]:
# =============================================================================
# BLOCK 8 — SUMMARY REPORT
# =============================================================================


def generate_summary_report(station_codes, title='FINAL SUMMARY'):
    """One row per station with the current row count of each product's CSV —
    the sanity check that every station has data from every product. Printed
    after the initial extraction, and again as the last thing the pipeline
    does, so the two are easy to compare."""
    rows = []
    for code in station_codes:
        code = str(code).replace('.0', '')
        row = {'station': code}
        for product, spec in PRODUCT_SPEC.items():
            row[product] = read_csv_row_count(station_folder(code) / spec['csv'])
        rows.append(row)
    report = pd.DataFrame(rows)
    log('=' * 80)
    log(f'{title} — rows per station per product')
    log('=' * 80)
    log(report.to_string(index=False))
    empty_products = {p: int((report[p].fillna(0) == 0).sum()) for p in PRODUCT_SPEC}
    log(f'Stations with zero rows, by product: {empty_products}')
    report.to_csv(OUTPUT_DIR / 'summary_report.csv', index=False)
    return report


# =============================================================================
# BLOCK 9 — MAIN
# =============================================================================
# Folder layout produced under OUTPUT_DIR (see the markdown cell for the
# full tree). Everything needed to resume a run — which granules were
# already handled, which dates were already checked, the CSVs themselves —
# lives on disk under each Station_<code>/ folder, so re-running this
# notebook against an environment that already has output picks up where it
# left off (nothing is re-downloaded or re-processed unnecessarily): each
# stage below logs how many rows/granules it already found before doing any
# work, so that resumption is visible, not just assumed.
#
# Processing runs one product at a time, across every station, rather than
# cycling through all four products station by station: each of Blocks 2-4
# is self-contained (its own search, download, and CSV), so finishing one
# product for every station before moving to the next keeps the run's
# progress easy to follow and avoids re-initializing per-product state
# (search patterns, discovered Raster tiles, etc.) on every switch.
#
# Order below and why: extract (2-4) -> completeness/recovery (6, needs all
# four products already on disk to compare) -> duplicate audit (7, the data
# should be clean before deriving anything from it) -> Reach WSE correction
# (5, needs the final, deduplicated Node/Reach data) -> summary.
#
# Every stage is gated by a flag, and STATION_CODES lets you restrict a run
# to a subset of stations (None = every station in STATIONS_VECTOR). Nothing
# here reads from stdin, so "Run All" always completes unattended.

STATION_CODES = None              # e.g. ['66090000', '66070004'] to restrict the run
RUN_DOWNLOAD_AND_PROCESS = True   # Blocks 2-4: search, download, extract WSE
RUN_COMPLETENESS_CHECK = True     # Block 6: cross-product missing-date recovery
COMPLETENESS_MAX_DATES_PER_CALL = 200  # per product, per call — re-run to work through a larger backlog
RUN_DUPLICATE_AUDIT = True        # Block 7: de-duplicate final CSVs
RUN_WSE_CORRECTION = True         # Block 5: Reach position correction


def _load_stations_and_masks():
    if not STATIONS_VECTOR.exists():
        raise FileNotFoundError(f'Stations layer not found: {STATIONS_VECTOR}. Set STATIONS_VECTOR in the config cell.')
    stations_gdf = gpd.read_file(STATIONS_VECTOR, engine=VECTOR_ENGINE)

    if WATER_MASK_VECTOR.exists():
        mask_gdf = gpd.read_file(WATER_MASK_VECTOR, engine=VECTOR_ENGINE)
    else:
        log(f'No water-mask layer at {WATER_MASK_VECTOR} -- falling back to a '
            f'{DEFAULT_SEARCH_BUFFER_KM} km buffer around each station point.')
        utm_crs = stations_gdf.estimate_utm_crs()
        buffered = stations_gdf.to_crs(utm_crs).copy()
        buffered['geometry'] = buffered.geometry.buffer(DEFAULT_SEARCH_BUFFER_KM * 1000)
        mask_gdf = buffered.to_crs(stations_gdf.crs)
    return stations_gdf, mask_gdf


def _station_geometry(code, mask_gdf, stations_gdf):
    """Per-station (bbox, water-mask GeoDataFrame, station-point GeoDataFrame),
    all three already sliced to just this station — the shape every one of
    the PIXC/Raster worker functions expects."""
    sel = mask_gdf[STATION_CODE_FIELD].astype(str).str.replace('.0', '') == code
    station_mask = mask_gdf[sel]
    if station_mask.empty:
        return None
    station_pt_rows = stations_gdf[stations_gdf[STATION_CODE_FIELD].astype(str).str.replace('.0', '') == code]
    station_point_geom = station_pt_rows.geometry.iloc[0] if not station_pt_rows.empty else station_mask.geometry.centroid.iloc[0]
    bbox = tuple(station_mask.to_crs('EPSG:4326').total_bounds)
    water_mask_gdf = gpd.GeoDataFrame(geometry=list(station_mask.geometry), crs=station_mask.crs)
    station_point_gdf = gpd.GeoDataFrame(geometry=[station_point_geom], crs=station_mask.crs)
    return bbox, water_mask_gdf, station_point_gdf


def main():
    log('=' * 80)
    log('SWOT multi-product WSE pipeline -- Pantanal Rivers')
    log('=' * 80)
    login_earthaccess()

    stations_gdf, mask_gdf = _load_stations_and_masks()
    all_codes = sorted(stations_gdf[STATION_CODE_FIELD].astype(str).str.replace('.0', '').unique())
    codes = [str(c).replace('.0', '') for c in STATION_CODES] if STATION_CODES else all_codes
    log(f'Stations to process: {len(codes)}')

    geometries = {code: _station_geometry(code, mask_gdf, stations_gdf) for code in codes}
    missing_geometry = [code for code, geom in geometries.items() if geom is None]
    if missing_geometry:
        log(f'No water mask/buffer available for {len(missing_geometry)} station(s), they will be skipped: {missing_geometry}')

    if RUN_DOWNLOAD_AND_PROCESS:
        log('\n' + '-' * 80 + '\n--- PIXC ---\n' + '-' * 80)
        for i, code in enumerate(codes, 1):
            if geometries[code] is None:
                continue
            bbox, water_mask_gdf, station_point_gdf = geometries[code]
            log(f'[{i}/{len(codes)}] station {code}')
            process_station_pixc(code, bbox, water_mask_gdf, station_point_gdf)
        generate_summary_report(codes, title='STAGE SUMMARY (after PIXC)')

        log('\n' + '-' * 80 + '\n--- Raster ---\n' + '-' * 80)
        for i, code in enumerate(codes, 1):
            if geometries[code] is None:
                continue
            bbox, water_mask_gdf, station_point_gdf = geometries[code]
            log(f'[{i}/{len(codes)}] station {code}')
            process_station_raster(code, bbox, water_mask_gdf, station_point_gdf)
        drop_low_pixel_count_rows(codes)
        generate_summary_report(codes, title='STAGE SUMMARY (after Raster)')

        log('\n' + '-' * 80 + '\n--- Node & Reach ---\n' + '-' * 80)
        for i, code in enumerate(codes, 1):
            if geometries[code] is None:
                continue
            bbox, _water_mask_gdf, _station_point_gdf = geometries[code]
            log(f'[{i}/{len(codes)}] station {code}')
            # Node/Reach filter the full layers internally (both feature types
            # need the station's code, not just its pre-sliced geometry).
            process_station_river(code, bbox, mask_gdf, stations_gdf)
        generate_summary_report(codes, title='STAGE SUMMARY (after Node/Reach)')

    if RUN_COMPLETENESS_CHECK:
        log('\n' + '-' * 80 + '\n--- Cross-product completeness check ---\n' + '-' * 80)
        for product in ('pixc', 'raster', 'node', 'reach'):
            run_completeness_check(product, codes, mask_gdf, stations_gdf, max_dates_total=COMPLETENESS_MAX_DATES_PER_CALL)

    if RUN_DUPLICATE_AUDIT:
        log('\n' + '-' * 80 + '\n--- Duplicate audit ---\n' + '-' * 80)
        run_duplicate_audit(codes)

    if RUN_WSE_CORRECTION:
        log('\n' + '-' * 80 + '\n--- Reach WSE position correction ---\n' + '-' * 80)
        compute_wse_corrected(codes, stations_gdf)

    generate_summary_report(codes, title='FINAL SUMMARY')
    log('\nPipeline run complete.')


main()
